## 1. FIND TITLE

In [59]:
import polars as pl
from pathlib import Path
import os
import resource
import time
import json
import psycopg
import climato.spikes.daily_wide_long as dwl
import climato.config as config
import climato.spikes.daily_wide_long_loader as loader

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
def current_rss_mb():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 1024


def peak_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


def print_memory_usage():
    print("PID:", os.getpid())
    print("RSS current:", current_rss_mb(), "MB")
    print("RSS peak:", peak_rss_mb(), "MB")


print_memory_usage()

PID: 2178955
RSS current: 93.01953125 MB
RSS peak: 92.25390625 MB


In [7]:
print("cwd:", Path.cwd())
print("module:", dwl.__file__)
print("DATA_DIR:", dwl.DATA_DIR)
print("DATA_DIR exists:", dwl.DATA_DIR.exists())

print("RTV matches:")
print(list(dwl.DATA_DIR.glob(dwl.RR_T_VENT_PATTERN)))

print("AP matches:")
print(list(dwl.DATA_DIR.glob(dwl.AUTRES_PARAMETRES_PATTERN)))

import sys

print(sys.executable)

cwd: /home/antoine/projects/personal/climato-lab/src/climato/spikes
module: /home/antoine/projects/personal/climato-lab/src/climato/spikes/daily_wide_long.py
DATA_DIR: /home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily
DATA_DIR exists: True
RTV matches:
[PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_avant-1949_RR-T-Vent.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_latest-2025-2026_RR-T-Vent.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_previous-1950-2024_RR-T-Vent.csv')]
AP matches:
[PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_previous-1950-2024_autres-parametres.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_avant-1949_autres-parametres.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_lates

In [8]:
def current_rss_mb():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 1024


def peak_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


def print_memory_usage():
    print("PID:", os.getpid())
    print("RSS current:", current_rss_mb(), "MB")
    print("RSS peak:", peak_rss_mb(), "MB")

The CSV source data files contain daily observations of meteorological parameters. Each parameter corresponds to a column in the files.
Each parameter value has a corresponding quality code. In the data files, quality codes are listed in a separate column. For a given parameter, the quality code is in the column immediately to the right of the column containing the parameter itself.

In [9]:
lf_rtv_source = dwl.scan_daily_rr_t_vent()
lf_rtv_wide_core = dwl.build_lf_rtv_wide_core(lf_rtv_source)
lf_rtv_wide_all = dwl.build_lf_rtv_wide_all(lf_rtv_source)

lf_ap_source = dwl.scan_daily_autres_parametres()
lf_ap_wide = dwl.build_lf_ap_wide(lf_ap_source)

In [10]:
DUMP_DIR = dwl.get_dump_dir()

## 2. Source data understanding

### 2.1. Corpus

The spike is currently limited to the daily MDGF data for department 79, the observed parameter coverage here is not assumed to be representative of the coverage at the national level, when all data are included.

In [30]:
families = {
    "RR-T-Vent": {
        "lf": lf_rtv_wide_all,
        "params": dwl.RR_T_VENT_ALL_PARAMS,
    },
    "autres-parametres": {
        "lf": lf_ap_wide,
        "params": dwl.AUTRES_PARAMETRES_ALL_PARAMS,
    },
}

family_summaries: list[pl.LazyFrame] = []
for family_name, family_dict in families.items():
    lf: pl.LazyFrame = family_dict["lf"]
    family_summaries.append(
        lf
        .select(
            source_family=pl.lit(family_name),
            files=pl.col("source_file").n_unique(),
            rows=pl.len(),
            stations=pl.col("station_name").n_unique(),
            parameters=len(family_dict["params"]),
            min_date=pl.col("obs_date").min(),
            max_date=pl.col("obs_date").max(),
        ),
    )

print(pl.concat(family_summaries).collect())

shape: (2, 7)
┌───────────────────┬───────┬─────────┬──────────┬────────────┬────────────┬────────────┐
│ source_family     ┆ files ┆ rows    ┆ stations ┆ parameters ┆ min_date   ┆ max_date   │
│ ---               ┆ ---   ┆ ---     ┆ ---      ┆ ---        ┆ ---        ┆ ---        │
│ str               ┆ u32   ┆ u32     ┆ u32      ┆ i32        ┆ date       ┆ date       │
╞═══════════════════╪═══════╪═════════╪══════════╪════════════╪════════════╪════════════╡
│ RR-T-Vent         ┆ 3     ┆ 1172316 ┆ 117      ┆ 26         ┆ 1865-07-01 ┆ 2026-08-12 │
│ autres-parametres ┆ 3     ┆ 565558  ┆ 70       ┆ 41         ┆ 1867-01-02 ┆ 2026-08-16 │
└───────────────────┴───────┴─────────┴──────────┴────────────┴────────────┴────────────┘


### 2.2. Parameter completeness

**TODO**
Parameter completeness -> try to identify groups based on count_pct

In [31]:
print("RR-T-Vent:")
with pl.Config(tbl_cols=20, tbl_rows=-1, tbl_width_chars=170):
    print(dwl.custom_describe(lf_rtv_wide_all))

RR-T-Vent:
shape: (58, 5)
┌──────────────┬─────────┬────────────┬───────────┬────────────────┐
│ column       ┆ count   ┆ null_count ┆ count_pct ┆ null_count_pct │
│ ---          ┆ ---     ┆ ---        ┆ ---       ┆ ---            │
│ str          ┆ u32     ┆ u32        ┆ f64       ┆ f64            │
╞══════════════╪═════════╪════════════╪═══════════╪════════════════╡
│ station_id   ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ station_name ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ obs_date     ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ source_file  ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ rr           ┆ 1169046 ┆ 3270       ┆ 99.72     ┆ 0.28           │
│ qrr          ┆ 1169115 ┆ 3201       ┆ 99.73     ┆ 0.27           │
│ tn           ┆ 437999  ┆ 734317     ┆ 37.36     ┆ 62.64          │
│ qtn          ┆ 439710  ┆ 732606     ┆ 37.51     ┆ 62.49          │
│ htn          ┆ 153736  ┆ 1018580    ┆ 13.11     ┆ 86.89          │
│ qhtn  

In [32]:
print("autres-parametres:")
with pl.Config(tbl_cols=20, tbl_rows=-1, tbl_width_chars=170):
    print(dwl.custom_describe(lf_ap_wide))

autres-parametres:
shape: (86, 5)
┌──────────────┬────────┬────────────┬───────────┬────────────────┐
│ column       ┆ count  ┆ null_count ┆ count_pct ┆ null_count_pct │
│ ---          ┆ ---    ┆ ---        ┆ ---       ┆ ---            │
│ str          ┆ u32    ┆ u32        ┆ f64       ┆ f64            │
╞══════════════╪════════╪════════════╪═══════════╪════════════════╡
│ station_id   ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ station_name ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ obs_date     ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ source_file  ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ dhumec       ┆ 7945   ┆ 557613     ┆ 1.4       ┆ 98.6           │
│ qdhumec      ┆ 8096   ┆ 557462     ┆ 1.43      ┆ 98.57          │
│ pmerm        ┆ 14801  ┆ 550757     ┆ 2.62      ┆ 97.38          │
│ qpmerm       ┆ 14801  ┆ 550757     ┆ 2.62      ┆ 97.38          │
│ pmermin      ┆ 6819   ┆ 558739     ┆ 1.21      ┆ 98.79          │
│ qpmermin    

### 2.3 Measurement density & recurring parameter signatures

#### 2.3.1. `RR-T-Vent`

**TODO**

The 1,172,316 wide rows actually contain 4,242,670 (non-null) measurements.  
Therefore:
- Each wide row has on average only approximately **3.62** measurements for the 26 parameters. 
- Measurement density is **13.9%** (= 4,242,670 / (1,172,316 * 26)).

In [ ]:
lf_rtv_profile = dwl.add_measure_count_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS, "all_measure_count")

summary = (
    lf_rtv_profile
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.RR_T_VENT_ALL_PARAMS)),
    )
    .collect()
)

print("RR-T-Vent all parameters:")
print(summary)

RR-T-Vent all parameters:
shape: (1, 4)
┌─────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows  ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---     ┆ ---        ┆ ---                        ┆ ---             │
│ u32     ┆ u32        ┆ f64                        ┆ f64             │
╞═════════╪════════════╪════════════════════════════╪═════════════════╡
│ 1172316 ┆ 4242670    ┆ 3.61905                    ┆ 0.139194        │
└─────────┴────────────┴────────────────────────────┴─────────────────┘


Below is the breakdown of wide rows by number of measured (non-null) parameters.

In [ ]:
with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_rtv_profile
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (23, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 427    ┆ 0.036424  │
│ 1                 ┆ 733810 ┆ 62.594898 │
│ 2                 ┆ 69     ┆ 0.005886  │
│ 3                 ┆ 4      ┆ 0.000341  │
│ 4                 ┆ 861    ┆ 0.073444  │
│ 5                 ┆ 280919 ┆ 23.962737 │
│ 6                 ┆ 1023   ┆ 0.087263  │
│ 7                 ┆ 731    ┆ 0.062355  │
│ 8                 ┆ 5882   ┆ 0.501742  │
│ 9                 ┆ 81105  ┆ 6.918356  │
│ 10                ┆ 206    ┆ 0.017572  │
│ 11                ┆ 153    ┆ 0.013051  │
│ 12                ┆ 42     ┆ 0.003583  │
│ 13                ┆ 49     ┆ 0.00418   │
│ 14                ┆ 28     ┆ 0.002388  │
│ 15                ┆ 91     ┆ 0.007762  │
│ 16                ┆ 2498   ┆ 0.213082  │
│ 17                ┆ 466    ┆ 0.03975 

Below is the breakdown of wide rows by the exact combination of measured parameters (signature).  
Of the 344 signature profiles observed on all `RR-T-Vent` rows:
- the most frequently observed accounts for approximately **62.6%** of all `RR-T-Vent` rows (rows with only `RR`)
- the 3 most frequently observed account for more than **93.4%** of all rows
- the 10 most frequently observed account for approximately **99.0%** of all rows 

In [ ]:
lf_rtv_profile = lf_rtv_profile.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_all = (
    lf_rtv_profile
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_rtv_all_head = patterns_rtv_all.head(30)
print(f"patterns_rtv_all.shape: {patterns_rtv_all.shape}")
print(f"patterns_rtv_all sum pct: {patterns_rtv_all.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_all_head)

patterns_rtv_all.shape: (344, 4)
patterns_rtv_all sum pct: 100.00000000000006
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 1                 ┆ RR                                                      ┆ 733800 ┆ 62.594045 │
│ 5                 ┆ RR,TN,TX,TNTXM,TAMPLI                                   ┆ 280908 ┆ 23.961799 │
│ 9                 ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG                     ┆ 81036  ┆ 6.912471  │
│ 19                ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG,FFM,FXY,DXY,HXY,FXI ┆ 23138  ┆ 1.9737    

#### 2.3.2. `autres-parametres`

**TODO**

The 565,558 wide rows actually contain 2,148,378 parameter measurements.  
Therefore:
- Each wide row has on average only approximately **3.80** measurements for the 41 parameters. 
- Measurement density is **9.3%** (= 2,148,378 / (565,558 * 41)).

In [ ]:
lf_ap_profile = dwl.add_measure_count_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS, "all_measure_count")

summary = (
    lf_ap_profile
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.AUTRES_PARAMETRES_ALL_PARAMS)),
    )
    .collect()
)

print("autres-parameters all parameters:")
print(summary)

autres-parameters all parameters:
shape: (1, 4)
┌────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---    ┆ ---        ┆ ---                        ┆ ---             │
│ u32    ┆ u32        ┆ f64                        ┆ f64             │
╞════════╪════════════╪════════════════════════════╪═════════════════╡
│ 565558 ┆ 2148378    ┆ 3.798687                   ┆ 0.092651        │
└────────┴────────────┴────────────────────────────┴─────────────────┘


Below is the breakdown of wide rows by number of measured (non-null) parameters (signature length).

In [ ]:
with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_ap_profile
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (30, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 370    ┆ 0.065422  │
│ 1                 ┆ 122836 ┆ 21.719435 │
│ 2                 ┆ 21626  ┆ 3.823834  │
│ 3                 ┆ 278177 ┆ 49.18629  │
│ 4                 ┆ 71335  ┆ 12.613207 │
│ 5                 ┆ 1198   ┆ 0.211826  │
│ 6                 ┆ 1330   ┆ 0.235166  │
│ 7                 ┆ 1741   ┆ 0.307838  │
│ 8                 ┆ 794    ┆ 0.140392  │
│ 9                 ┆ 39317  ┆ 6.951895  │
│ 10                ┆ 306    ┆ 0.054106  │
│ 11                ┆ 7693   ┆ 1.36025   │
│ 12                ┆ 1256   ┆ 0.222082  │
│ 13                ┆ 2783   ┆ 0.49208   │
│ 14                ┆ 78     ┆ 0.013792  │
│ 15                ┆ 372    ┆ 0.065776  │
│ 16                ┆ 255    ┆ 0.045088  │
│ 17                ┆ 671    ┆ 0.118644

Below is the breakdown of wide rows by the exact combination of measured parameters (signature).  
Of the 635 signature profiles observed on all `autres-parametres` rows:
- the most frequently observed accounts for more than **47.8%** of all `autres-parametres` rows (signature length 3: rows with only `BROU`, `ORAG` and `GRELE`)
- the 3 most frequently observed account for approximately **79.9%** of all rows (signature length <= 4)
- the 10 most frequently observed account for more than **92.5%** of all rows (signature length <= 11)

In [ ]:
lf_ap_profile = lf_ap_profile.with_columns(
    dwl.measure_signature_expr(dwl.AUTRES_PARAMETRES_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_ap = (
    lf_ap_profile
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_ap_head = patterns_ap.head(30)
print(f"patterns_ap.shape: {patterns_ap.shape}")
print(f"patterns_ap, sum pct: {patterns_ap.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_ap_head)

patterns_ap.shape: (635, 4)
patterns_ap, sum pct: 99.99999999999996
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 3                 ┆ BROU,ORAG,GRELE                                         ┆ 270391 ┆ 47.809597 │
│ 1                 ┆ ETPGRILLE                                               ┆ 114089 ┆ 20.17282  │
│ 4                 ┆ ETPGRILLE,BROU,ORAG,GRELE                               ┆ 67175  ┆ 11.87765  │
│ 9                 ┆ UN,HUN,UX,HUX,UM,DHUMI40,DHUMI80,TSVM,ETPGRILLE         ┆ 34402  ┆ 6.082842  │
│ 2     

### DELETE 2.4 Recurring parameter signatures

#### 2.4.1 `RR-T-Vent`

##### 2.4.1.1. Core parameters

If we consider only 5 core parameters (`RR`, `TN`, `TX`, `TM`, `TNTXM`) out of the 26 in the `RR-T-Vent` files, we can observe that of the 1,172,316 lines:
- 62.60% have 1 measurement (most frequently `RR`)
- 24.25% have 4 measurements (most frequently the combination of `RR`, `TN`, `TX`, `TNTXM`)
- 13.03% have 5 measurements (all 5 core parameters)  

They account for virtually all of the DataFrame's lines (99.88%).

We have the following statistics:
- Number of wide rows: 1,172,316
- Expected number of long (core) rows (after unpivoting): 2,637,628
- Mean number of (core) measures per wide row: 2.2499
- (Core) measurement density: 45.00% (= 2.2499 / 5)

In [29]:
lf_rtv_wide_core = lf_rtv_wide_core.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_CORE_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_core = (
    lf_rtv_wide_core
    .group_by(
        "core_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_core.head(30))

ColumnNotFoundError: unable to find column "core_measure_count"; valid columns: ["station_id", "station_name", "obs_date", "source_file", "rr", "qrr", "tn", "qtn", "tx", "qtx", "tm", "qtm", "tntxm", "qtntxm", "measure_signature"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
 WITH_COLUMNS:
 [when(col("rr").is_not_null()).then("RR").otherwise(null).str.concat_horizontal([when(col("tn").is_not_null()).then("TN").otherwise(null), when(col("tx").is_not_null()).then("TX").otherwise(null), when(col("tm").is_not_null()).then("TM").otherwise(null), when(col("tntxm").is_not_null()).then("TNTXM").otherwise(null)]).alias("measure_signature")] 
  SELECT [col("NUM_POSTE").alias("station_id"), col("NOM_USUEL").alias("station_name"), col("obs_date"), col("source_file"), col("RR").alias("rr"), col("QRR").alias("qrr"), col("TN").alias("tn"), col("QTN").alias("qtn"), col("TX").alias("tx"), col("QTX").alias("qtx"), col("TM").alias("tm"), col("QTM").alias("qtm"), col("TNTXM").al

##### 2.4.1.2. All `RR-T-Vent` parameters

#### 2.4.2. `autres-parametres`

### 2.4 Value / quality consistency

#### 2.4.1. `RR-T-Vent`

##### 2.4.1.1. Audit of parameter / quality code pairs

In [ ]:
rtv_all_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(rtv_all_quality_audit)

shape: (52, 2)
┌──────────────────────────────┬───────┐
│ line_type                    ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ RR_value_without_quality     ┆ 0     │
│ RR_quality_without_value     ┆ 69    │
│ TN_value_without_quality     ┆ 0     │
│ TN_quality_without_value     ┆ 1711  │
│ HTN_value_without_quality    ┆ 0     │
│ HTN_quality_without_value    ┆ 2066  │
│ TX_value_without_quality     ┆ 0     │
│ TX_quality_without_value     ┆ 1705  │
│ HTX_value_without_quality    ┆ 0     │
│ HTX_quality_without_value    ┆ 2082  │
│ TM_value_without_quality     ┆ 0     │
│ TM_quality_without_value     ┆ 1714  │
│ TNTXM_value_without_quality  ┆ 0     │
│ TNTXM_quality_without_value  ┆ 1     │
│ TAMPLI_value_without_quality ┆ 0     │
│ TAMPLI_quality_without_value ┆ 1     │
│ TNSOL_value_without_quality  ┆ 0     │
│ TNSOL_quality_without_value  ┆ 24    │
│ TN50_value_without_quality   ┆ 0     │
│

In [ ]:
all_quality_pairs_metadata, all_quality_pairs_expressions = dwl.quality_pair_audit_lf_2(lf=None, params=dwl.RR_T_VENT_ALL_PARAMS)
all_quality_pairs_counts = (
    lf_rtv_wide_all
    .select(all_quality_pairs_expressions)
    .collect()
)
print(all_quality_pairs_counts)

shape: (1, 52)
┌──────────┬──────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ metric_0 ┆ metric_1 ┆ metric_2 ┆ metric_3 ┆ … ┆ metric_48 ┆ metric_49 ┆ metric_50 ┆ metric_51 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32      ┆ u32      ┆ u32      ┆ u32      ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞══════════╪══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 69       ┆ 0        ┆ 1711     ┆ … ┆ 0         ┆ 17        ┆ 0         ┆ 1644      │
└──────────┴──────────┴──────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [ ]:
all_quality_pairs_counts_long = all_quality_pairs_counts.unpivot(
    variable_name="metric_id",
    value_name="count",
)
print(all_quality_pairs_counts_long)

shape: (52, 2)
┌───────────┬───────┐
│ metric_id ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ metric_0  ┆ 0     │
│ metric_1  ┆ 69    │
│ metric_2  ┆ 0     │
│ metric_3  ┆ 1711  │
│ metric_4  ┆ 0     │
│ …         ┆ …     │
│ metric_47 ┆ 17    │
│ metric_48 ┆ 0     │
│ metric_49 ┆ 17    │
│ metric_50 ┆ 0     │
│ metric_51 ┆ 1644  │
└───────────┴───────┘


In [ ]:
all_quality_pairs_metadata_df = pl.DataFrame(all_quality_pairs_metadata)
print(all_quality_pairs_metadata_df)

shape: (52, 3)
┌───────────┬───────────────────────┬───────┐
│ metric_id ┆ pair_type             ┆ param │
│ ---       ┆ ---                   ┆ ---   │
│ str       ┆ str                   ┆ str   │
╞═══════════╪═══════════════════════╪═══════╡
│ metric_0  ┆ value_without_quality ┆ RR    │
│ metric_1  ┆ quality_without_value ┆ RR    │
│ metric_2  ┆ value_without_quality ┆ TN    │
│ metric_3  ┆ quality_without_value ┆ TN    │
│ metric_4  ┆ value_without_quality ┆ HTN   │
│ …         ┆ …                     ┆ …     │
│ metric_47 ┆ quality_without_value ┆ DXI3S │
│ metric_48 ┆ value_without_quality ┆ HXI3S │
│ metric_49 ┆ quality_without_value ┆ HXI3S │
│ metric_50 ┆ value_without_quality ┆ DRR   │
│ metric_51 ┆ quality_without_value ┆ DRR   │
└───────────┴───────────────────────┴───────┘


In [ ]:
all_quality_pairs_rtv_audit = (
    all_quality_pairs_counts_long
    .join(
        all_quality_pairs_metadata_df,
        how="left",
        on="metric_id",
    )
)
print(all_quality_pairs_rtv_audit)

shape: (52, 4)
┌───────────┬───────┬───────────────────────┬───────┐
│ metric_id ┆ count ┆ pair_type             ┆ param │
│ ---       ┆ ---   ┆ ---                   ┆ ---   │
│ str       ┆ u32   ┆ str                   ┆ str   │
╞═══════════╪═══════╪═══════════════════════╪═══════╡
│ metric_0  ┆ 0     ┆ value_without_quality ┆ RR    │
│ metric_1  ┆ 69    ┆ quality_without_value ┆ RR    │
│ metric_2  ┆ 0     ┆ value_without_quality ┆ TN    │
│ metric_3  ┆ 1711  ┆ quality_without_value ┆ TN    │
│ metric_4  ┆ 0     ┆ value_without_quality ┆ HTN   │
│ …         ┆ …     ┆ …                     ┆ …     │
│ metric_47 ┆ 17    ┆ quality_without_value ┆ DXI3S │
│ metric_48 ┆ 0     ┆ value_without_quality ┆ HXI3S │
│ metric_49 ┆ 17    ┆ quality_without_value ┆ HXI3S │
│ metric_50 ┆ 0     ┆ value_without_quality ┆ DRR   │
│ metric_51 ┆ 1644  ┆ quality_without_value ┆ DRR   │
└───────────┴───────┴───────────────────────┴───────┘


In [ ]:
all_quality_pairs_rtv_matrix = (
    all_quality_pairs_rtv_audit
    .pivot(
        on="pair_type",
        index="param",
        values="count",
    )
)
with pl.Config(tbl_rows=30):
    print(all_quality_pairs_rtv_matrix)

shape: (26, 3)
┌────────┬───────────────────────┬───────────────────────┐
│ param  ┆ value_without_quality ┆ quality_without_value │
│ ---    ┆ ---                   ┆ ---                   │
│ str    ┆ u32                   ┆ u32                   │
╞════════╪═══════════════════════╪═══════════════════════╡
│ RR     ┆ 0                     ┆ 69                    │
│ TN     ┆ 0                     ┆ 1711                  │
│ HTN    ┆ 0                     ┆ 2066                  │
│ TX     ┆ 0                     ┆ 1705                  │
│ HTX    ┆ 0                     ┆ 2082                  │
│ TM     ┆ 0                     ┆ 1714                  │
│ TNTXM  ┆ 0                     ┆ 1                     │
│ TAMPLI ┆ 0                     ┆ 1                     │
│ TNSOL  ┆ 0                     ┆ 24                    │
│ TN50   ┆ 0                     ┆ 0                     │
│ DG     ┆ 0                     ┆ 1761                  │
│ FFM    ┆ 0                     ┆ 38    

##### 2.4.1.2. Orphan quality codes audit

###### 2.4.1.2.1 Orphan quality codes audit. Slow version (`concat`)

In [ ]:
audit_parts: list[pl.LazyFrame] = []

for param in dwl.RR_T_VENT_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_all = audit_parts.append(
        lf_rtv_wide_all
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
    )

orphan_quality_audit_rtv_all_concat = pl.concat(audit_parts).collect()

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_all_concat)

shape: (39, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qhtn           ┆ 1                  ┆ 2032 │
│ qhtn           ┆ 9                  ┆ 34   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qhtx           ┆ 1                  ┆ 2051 │
│ qhtx           ┆ 9                  ┆ 31   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
│ qtampli        ┆ 1                  ┆ 1    │
│ qtnsol         ┆ 1                  ┆ 15   │
│ qtnsol         ┆ 9                  ┆ 9    │
│ qdg            ┆ 1                  ┆ 1751 

In [ ]:
quality_columns = pl.DataFrame({
    "quality_column": [f"q{param.lower()}" for param in dwl.RR_T_VENT_ALL_PARAMS],
})

quality_codes = pl.DataFrame({
    "quality_code_value": [0, 1, 2, 9],
})

quality_combinations = quality_columns.join(quality_codes, how="cross")

print(quality_combinations)

q_join = (
    quality_combinations
    .join(
        orphan_quality_audit_rtv_all_concat,
        how="left",
        on=["quality_column", "quality_code_value"],
    )
    .fill_null(0)
)

qm = q_join.pivot(
    on="quality_code_value",
    index="quality_column",
    values="len",
)

with pl.Config(tbl_rows=150):
    print(qm)

shape: (104, 2)
┌────────────────┬────────────────────┐
│ quality_column ┆ quality_code_value │
│ ---            ┆ ---                │
│ str            ┆ i64                │
╞════════════════╪════════════════════╡
│ qrr            ┆ 0                  │
│ qrr            ┆ 1                  │
│ qrr            ┆ 2                  │
│ qrr            ┆ 9                  │
│ qtn            ┆ 0                  │
│ …              ┆ …                  │
│ qhxi3s         ┆ 9                  │
│ qdrr           ┆ 0                  │
│ qdrr           ┆ 1                  │
│ qdrr           ┆ 2                  │
│ qdrr           ┆ 9                  │
└────────────────┴────────────────────┘
shape: (26, 5)
┌────────────────┬─────┬──────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1    ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---  ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32  ┆ u32 ┆ u32 │
╞════════════════╪═════╪══════╪═════╪═════╡
│ qrr            ┆ 0   ┆ 69   ┆ 0   ┆ 0   │
│ qtn            ┆ 0   ┆ 

###### 2.4.1.2.2 Orphan quality codes audit. Faster version (`select` expressions + `unpivot`)

In [ ]:
QUALITY_CODES = (0, 1, 2, 9)

audit_expressions = []
audit_metadata = []

for param in dwl.RR_T_VENT_ALL_PARAMS:
    param_col = param.lower()
    quality_col = f"q{param_col}"

    for quality_code in QUALITY_CODES:
        metric_id = f"metric_{len(audit_metadata)}"

        audit_expressions.append(
            (
                pl.col(param_col).is_null()
                & (pl.col(quality_col) == quality_code)
            )
            .sum()
            .alias(metric_id)
        )

        audit_metadata.append({
            "metric_id": metric_id,
            "quality_column": quality_col,
            "quality_code_value": quality_code,
        })

audit_counts = (
    lf_rtv_wide_all
    .select(audit_expressions)
    .collect()
)

In [ ]:
audit_metadata

[{'metric_id': 'metric_0', 'quality_column': 'qrr', 'quality_code_value': 0},
 {'metric_id': 'metric_1', 'quality_column': 'qrr', 'quality_code_value': 1},
 {'metric_id': 'metric_2', 'quality_column': 'qrr', 'quality_code_value': 2},
 {'metric_id': 'metric_3', 'quality_column': 'qrr', 'quality_code_value': 9},
 {'metric_id': 'metric_4', 'quality_column': 'qtn', 'quality_code_value': 0},
 {'metric_id': 'metric_5', 'quality_column': 'qtn', 'quality_code_value': 1},
 {'metric_id': 'metric_6', 'quality_column': 'qtn', 'quality_code_value': 2},
 {'metric_id': 'metric_7', 'quality_column': 'qtn', 'quality_code_value': 9},
 {'metric_id': 'metric_8', 'quality_column': 'qhtn', 'quality_code_value': 0},
 {'metric_id': 'metric_9', 'quality_column': 'qhtn', 'quality_code_value': 1},
 {'metric_id': 'metric_10', 'quality_column': 'qhtn', 'quality_code_value': 2},
 {'metric_id': 'metric_11', 'quality_column': 'qhtn', 'quality_code_value': 9},
 {'metric_id': 'metric_12', 'quality_column': 'qtx', 'qual

In [ ]:
audit_counts_long = audit_counts.unpivot(
    variable_name="metric_id",
    value_name="len",
)
with pl.Config(tbl_rows=150):
    print(audit_counts_long)

shape: (104, 2)
┌────────────┬──────┐
│ metric_id  ┆ len  │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ metric_0   ┆ 0    │
│ metric_1   ┆ 69   │
│ metric_2   ┆ 0    │
│ metric_3   ┆ 0    │
│ metric_4   ┆ 0    │
│ metric_5   ┆ 1683 │
│ metric_6   ┆ 0    │
│ metric_7   ┆ 28   │
│ metric_8   ┆ 0    │
│ metric_9   ┆ 2032 │
│ metric_10  ┆ 0    │
│ metric_11  ┆ 34   │
│ metric_12  ┆ 0    │
│ metric_13  ┆ 1687 │
│ metric_14  ┆ 0    │
│ metric_15  ┆ 18   │
│ metric_16  ┆ 0    │
│ metric_17  ┆ 2051 │
│ metric_18  ┆ 0    │
│ metric_19  ┆ 31   │
│ metric_20  ┆ 0    │
│ metric_21  ┆ 1711 │
│ metric_22  ┆ 0    │
│ metric_23  ┆ 3    │
│ metric_24  ┆ 0    │
│ metric_25  ┆ 1    │
│ metric_26  ┆ 0    │
│ metric_27  ┆ 0    │
│ metric_28  ┆ 0    │
│ metric_29  ┆ 1    │
│ metric_30  ┆ 0    │
│ metric_31  ┆ 0    │
│ metric_32  ┆ 0    │
│ metric_33  ┆ 15   │
│ metric_34  ┆ 0    │
│ metric_35  ┆ 9    │
│ metric_36  ┆ 0    │
│ metric_37  ┆ 0    │
│ metric_38  ┆ 0    │
│ metric_39  ┆ 0

In [ ]:
audit_metadata_df = pl.DataFrame(
    audit_metadata,
    schema_overrides={
        "quality_code_value": pl.UInt8,
    },
)
with pl.Config(tbl_rows=150):
    print(audit_metadata_df)

shape: (104, 3)
┌────────────┬────────────────┬────────────────────┐
│ metric_id  ┆ quality_column ┆ quality_code_value │
│ ---        ┆ ---            ┆ ---                │
│ str        ┆ str            ┆ u8                 │
╞════════════╪════════════════╪════════════════════╡
│ metric_0   ┆ qrr            ┆ 0                  │
│ metric_1   ┆ qrr            ┆ 1                  │
│ metric_2   ┆ qrr            ┆ 2                  │
│ metric_3   ┆ qrr            ┆ 9                  │
│ metric_4   ┆ qtn            ┆ 0                  │
│ metric_5   ┆ qtn            ┆ 1                  │
│ metric_6   ┆ qtn            ┆ 2                  │
│ metric_7   ┆ qtn            ┆ 9                  │
│ metric_8   ┆ qhtn           ┆ 0                  │
│ metric_9   ┆ qhtn           ┆ 1                  │
│ metric_10  ┆ qhtn           ┆ 2                  │
│ metric_11  ┆ qhtn           ┆ 9                  │
│ metric_12  ┆ qtx            ┆ 0                  │
│ metric_13  ┆ qtx            

In [ ]:
orphan_quality_audit_rtv_all = (
    audit_counts_long
    .join(
        audit_metadata_df,
        how="left",
        on="metric_id",
        validate="1:1",
    )
)

with pl.Config(tbl_rows=150):
    print(orphan_quality_audit_rtv_all)

shape: (104, 4)
┌────────────┬──────┬────────────────┬────────────────────┐
│ metric_id  ┆ len  ┆ quality_column ┆ quality_code_value │
│ ---        ┆ ---  ┆ ---            ┆ ---                │
│ str        ┆ u32  ┆ str            ┆ u8                 │
╞════════════╪══════╪════════════════╪════════════════════╡
│ metric_0   ┆ 0    ┆ qrr            ┆ 0                  │
│ metric_1   ┆ 69   ┆ qrr            ┆ 1                  │
│ metric_2   ┆ 0    ┆ qrr            ┆ 2                  │
│ metric_3   ┆ 0    ┆ qrr            ┆ 9                  │
│ metric_4   ┆ 0    ┆ qtn            ┆ 0                  │
│ metric_5   ┆ 1683 ┆ qtn            ┆ 1                  │
│ metric_6   ┆ 0    ┆ qtn            ┆ 2                  │
│ metric_7   ┆ 28   ┆ qtn            ┆ 9                  │
│ metric_8   ┆ 0    ┆ qhtn           ┆ 0                  │
│ metric_9   ┆ 2032 ┆ qhtn           ┆ 1                  │
│ metric_10  ┆ 0    ┆ qhtn           ┆ 2                  │
│ metric_11  ┆ 34   ┆ qh

In [ ]:
orphan_quality_matrix_rtv_all = orphan_quality_audit_rtv_all.pivot(
    on="quality_code_value",
    index="quality_column",
    values="len",
    sort_columns=True,
)
with pl.Config(tbl_rows=30):
    print(orphan_quality_matrix_rtv_all)

shape: (26, 5)
┌────────────────┬─────┬──────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1    ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---  ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32  ┆ u32 ┆ u32 │
╞════════════════╪═════╪══════╪═════╪═════╡
│ qrr            ┆ 0   ┆ 69   ┆ 0   ┆ 0   │
│ qtn            ┆ 0   ┆ 1683 ┆ 0   ┆ 28  │
│ qhtn           ┆ 0   ┆ 2032 ┆ 0   ┆ 34  │
│ qtx            ┆ 0   ┆ 1687 ┆ 0   ┆ 18  │
│ qhtx           ┆ 0   ┆ 2051 ┆ 0   ┆ 31  │
│ qtm            ┆ 0   ┆ 1711 ┆ 0   ┆ 3   │
│ qtntxm         ┆ 0   ┆ 1    ┆ 0   ┆ 0   │
│ qtampli        ┆ 0   ┆ 1    ┆ 0   ┆ 0   │
│ qtnsol         ┆ 0   ┆ 15   ┆ 0   ┆ 9   │
│ qtn50          ┆ 0   ┆ 0    ┆ 0   ┆ 0   │
│ qdg            ┆ 0   ┆ 1751 ┆ 0   ┆ 10  │
│ qffm           ┆ 0   ┆ 37   ┆ 0   ┆ 1   │
│ qff2m          ┆ 0   ┆ 0    ┆ 0   ┆ 0   │
│ qfxy           ┆ 0   ┆ 40   ┆ 0   ┆ 1   │
│ qdxy           ┆ 0   ┆ 41   ┆ 0   ┆ 1   │
│ qhxy           ┆ 0   ┆ 42   ┆ 0   ┆ 1   │
│ qfxi           ┆ 0   ┆ 37   ┆ 0   ┆ 1   │
│ qdxi           

#### 2.4.2. `autres-parametres`

##### 2.4.2.1. Audit of parameter / quality code pairs

In [ ]:
all_quality_pairs_metadata, all_quality_pairs_expressions = dwl.quality_pair_audit_lf_2(lf=None, params=dwl.AUTRES_PARAMETRES_ALL_PARAMS)
all_quality_pairs_counts = (
    lf_ap_wide
    .select(all_quality_pairs_expressions)
    .collect()
)
print(all_quality_pairs_counts)

shape: (1, 82)
┌──────────┬──────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ metric_0 ┆ metric_1 ┆ metric_2 ┆ metric_3 ┆ … ┆ metric_78 ┆ metric_79 ┆ metric_80 ┆ metric_81 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32      ┆ u32      ┆ u32      ┆ u32      ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞══════════╪══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 151      ┆ 0        ┆ 0        ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
└──────────┴──────────┴──────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [ ]:
all_quality_pairs_counts_long = (
    all_quality_pairs_counts
    .unpivot(
        variable_name="metric_id",
        value_name="count",
    )
)
print(all_quality_pairs_counts_long)

shape: (82, 2)
┌───────────┬───────┐
│ metric_id ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ metric_0  ┆ 0     │
│ metric_1  ┆ 151   │
│ metric_2  ┆ 0     │
│ metric_3  ┆ 0     │
│ metric_4  ┆ 0     │
│ …         ┆ …     │
│ metric_77 ┆ 0     │
│ metric_78 ┆ 0     │
│ metric_79 ┆ 0     │
│ metric_80 ┆ 0     │
│ metric_81 ┆ 0     │
└───────────┴───────┘


In [ ]:
all_quality_pairs_metadata_df = pl.DataFrame(all_quality_pairs_metadata)
print(all_quality_pairs_metadata_df)

shape: (82, 3)
┌───────────┬───────────────────────┬─────────┐
│ metric_id ┆ pair_type             ┆ param   │
│ ---       ┆ ---                   ┆ ---     │
│ str       ┆ str                   ┆ str     │
╞═══════════╪═══════════════════════╪═════════╡
│ metric_0  ┆ value_without_quality ┆ DHUMEC  │
│ metric_1  ┆ quality_without_value ┆ DHUMEC  │
│ metric_2  ┆ value_without_quality ┆ PMERM   │
│ metric_3  ┆ quality_without_value ┆ PMERM   │
│ metric_4  ┆ value_without_quality ┆ PMERMIN │
│ …         ┆ …                     ┆ …       │
│ metric_77 ┆ quality_without_value ┆ BA300   │
│ metric_78 ┆ value_without_quality ┆ TMERMIN │
│ metric_79 ┆ quality_without_value ┆ TMERMIN │
│ metric_80 ┆ value_without_quality ┆ TMERMAX │
│ metric_81 ┆ quality_without_value ┆ TMERMAX │
└───────────┴───────────────────────┴─────────┘


In [ ]:
all_quality_pairs_ap_audit = (
    all_quality_pairs_counts_long
    .join(
        all_quality_pairs_metadata_df,
        on="metric_id",
        how="left",
        validate="1:1",
    )
)
print(all_quality_pairs_ap_audit)

shape: (82, 4)
┌───────────┬───────┬───────────────────────┬─────────┐
│ metric_id ┆ count ┆ pair_type             ┆ param   │
│ ---       ┆ ---   ┆ ---                   ┆ ---     │
│ str       ┆ u32   ┆ str                   ┆ str     │
╞═══════════╪═══════╪═══════════════════════╪═════════╡
│ metric_0  ┆ 0     ┆ value_without_quality ┆ DHUMEC  │
│ metric_1  ┆ 151   ┆ quality_without_value ┆ DHUMEC  │
│ metric_2  ┆ 0     ┆ value_without_quality ┆ PMERM   │
│ metric_3  ┆ 0     ┆ quality_without_value ┆ PMERM   │
│ metric_4  ┆ 0     ┆ value_without_quality ┆ PMERMIN │
│ …         ┆ …     ┆ …                     ┆ …       │
│ metric_77 ┆ 0     ┆ quality_without_value ┆ BA300   │
│ metric_78 ┆ 0     ┆ value_without_quality ┆ TMERMIN │
│ metric_79 ┆ 0     ┆ quality_without_value ┆ TMERMIN │
│ metric_80 ┆ 0     ┆ value_without_quality ┆ TMERMAX │
│ metric_81 ┆ 0     ┆ quality_without_value ┆ TMERMAX │
└───────────┴───────┴───────────────────────┴─────────┘


In [ ]:
all_quality_pairs_ap_matrix = (
    all_quality_pairs_ap_audit
    .pivot(
        on="pair_type",
        index="param",
        values="count",
    )
)
with pl.Config(tbl_rows=50):
    print(all_quality_pairs_ap_matrix)

shape: (41, 3)
┌─────────────┬───────────────────────┬───────────────────────┐
│ param       ┆ value_without_quality ┆ quality_without_value │
│ ---         ┆ ---                   ┆ ---                   │
│ str         ┆ u32                   ┆ u32                   │
╞═════════════╪═══════════════════════╪═══════════════════════╡
│ DHUMEC      ┆ 0                     ┆ 151                   │
│ PMERM       ┆ 0                     ┆ 0                     │
│ PMERMIN     ┆ 0                     ┆ 2                     │
│ INST        ┆ 0                     ┆ 0                     │
│ GLOT        ┆ 0                     ┆ 5                     │
│ DIFT        ┆ 0                     ┆ 0                     │
│ DIRT        ┆ 0                     ┆ 0                     │
│ INFRART     ┆ 0                     ┆ 0                     │
│ UV          ┆ 0                     ┆ 0                     │
│ UV_INDICEX  ┆ 0                     ┆ 0                     │
│ SIGMA       ┆ 0        

##### 2.4.2.2. Orphan quality codes audit

In [ ]:
QUALITY_CODES = (0, 1, 2, 9)

audit_expressions = []
audit_metadata = []

for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS:
    param_col = param.lower()
    quality_col = f"q{param_col}"

    for quality_code in QUALITY_CODES:
        metric_id = f"metric_{len(audit_metadata)}"

        audit_expressions.append(
            (
                pl.col(param_col).is_null()
                & (pl.col(quality_col) == quality_code)
            )
            .sum()
            .alias(metric_id)
        )

        audit_metadata.append({
            "metric_id": metric_id,
            "quality_column": quality_col,
            "quality_code_value": quality_code,
        })

audit_counts = (
    lf_ap_wide
    .select(audit_expressions)
    .collect()
)

audit_counts_long = (
    audit_counts
    .unpivot(
        variable_name="metric_id",
        value_name="count",
    )
)

audit_metadata_df = pl.DataFrame(
    audit_metadata,
    schema_overrides={
        "quality_code_value": pl.UInt8,
    }
)

orphan_quality_audit_ap = (
    audit_counts_long
    .join(
        audit_metadata_df,
        on="metric_id",
        how="left",
        validate="1:1",
    )
)

orphan_quality_matrix_ap = (
    orphan_quality_audit_ap
    .pivot(
        on="quality_code_value",
        index="quality_column",
        values="count",
    )
)

with pl.Config(tbl_rows=50):
    print(orphan_quality_matrix_ap)

shape: (41, 5)
┌────────────────┬─────┬───────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1     ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---   ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32   ┆ u32 ┆ u32 │
╞════════════════╪═════╪═══════╪═════╪═════╡
│ qdhumec        ┆ 0   ┆ 151   ┆ 0   ┆ 0   │
│ qpmerm         ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qpmermin       ┆ 0   ┆ 1     ┆ 0   ┆ 1   │
│ qinst          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qglot          ┆ 0   ┆ 5     ┆ 0   ┆ 0   │
│ qdift          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qdirt          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qinfrart       ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ quv            ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ quv_indicex    ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qsigma         ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qun            ┆ 0   ┆ 44    ┆ 0   ┆ 1   │
│ qhun           ┆ 0   ┆ 82    ┆ 0   ┆ 3   │
│ qux            ┆ 0   ┆ 27    ┆ 0   ┆ 1   │
│ qhux           ┆ 0   ┆ 45    ┆ 0   ┆ 1   │
│ qum            ┆ 0   ┆ 28    ┆ 0   ┆ 2   │
│ qdhumi40       ┆ 0   ┆ 54    ┆ 0   ┆ 2

##### 2.4.2.3. Drill-down on orphan quality codes for `ETPMON` and `NEIGETOTX`

In [ ]:
with pl.Config(tbl_rows=20):
    print(dwl.drill_down_orphan_quality_pairs(lf_ap_wide, param="ETPMON", quality_code_value=1))

shape: (11, 5)
┌────────────┬──────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name         ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                  ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                  ┆ u32   ┆ date       ┆ date       │
╞════════════╪══════════════════════╪═══════╪════════════╪════════════╡
│ 79049004   ┆ BRESSUIRE            ┆ 8774  ┆ 1990-11-10 ┆ 2026-04-07 │
│ 79134001   ┆ GLENAY_SAPC          ┆ 1238  ┆ 1993-02-01 ┆ 1996-07-31 │
│ 79174002   ┆ MELLE                ┆ 5774  ┆ 1990-02-18 ┆ 2026-04-05 │
│ 79191005   ┆ NIORT                ┆ 38    ┆ 1991-12-28 ┆ 2018-12-14 │
│ 79202003   ┆ PARTHENAY_SAPC       ┆ 1035  ┆ 1993-09-23 ┆ 1996-07-31 │
│ 79215001   ┆ POUGNE-HERISSON_SAPC ┆ 1267  ┆ 1992-10-02 ┆ 1996-07-31 │
│ 79309001   ┆ SCILLE               ┆ 7299  ┆ 2006-02-10 ┆ 2026-04-07 │
│ 79326004   ┆ THENEZAY STNA        ┆ 8699  ┆ 2002-05-29 ┆ 2026-04-07 │
│ 79329001   ┆ THOUARS-STNA         ┆ 3386  ┆ 199

In [ ]:
with pl.Config(tbl_rows=20):
    print(dwl.drill_down_orphan_quality_pairs(lf_ap_wide, param="NEIGETOTX", quality_code_value=1))

shape: (10, 5)
┌────────────┬──────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name         ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                  ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                  ┆ u32   ┆ date       ┆ date       │
╞════════════╪══════════════════════╪═══════╪════════════╪════════════╡
│ 79049004   ┆ BRESSUIRE            ┆ 842   ┆ 1991-04-24 ┆ 1997-11-25 │
│ 79134001   ┆ GLENAY_SAPC          ┆ 835   ┆ 1994-05-30 ┆ 1999-11-21 │
│ 79174002   ┆ MELLE                ┆ 865   ┆ 1990-05-15 ┆ 1999-12-16 │
│ 79191005   ┆ NIORT                ┆ 2609  ┆ 1986-01-01 ┆ 1999-12-31 │
│ 79202003   ┆ PARTHENAY_SAPC       ┆ 978   ┆ 1993-10-01 ┆ 1999-11-07 │
│ 79215001   ┆ POUGNE-HERISSON_SAPC ┆ 1263  ┆ 1992-07-01 ┆ 1999-02-13 │
│ 79253002   ┆ LA TOUCHE STNA       ┆ 268   ┆ 1996-10-08 ┆ 1999-10-06 │
│ 79329001   ┆ THOUARS-STNA         ┆ 1008  ┆ 1993-09-24 ┆ 1998-07-29 │
│ 79342002   ┆ VERNOUX STNA         ┆ 772   ┆ 199

### 2.5 Audit of lines without measurements

#### 2.5.1 `RR-T-Vent`

In [ ]:
print(
    lf_rtv_profile
    .filter(
        pl.col("all_measure_count") == 0
    )
    .select(
        "station_id",
        "station_name",
        "obs_date"
    )
    .group_by(
        "station_id",
        "station_name",
    )
    .agg(
        count=pl.len(),
        min_date=pl.col("obs_date").min(),
        max_date=pl.col("obs_date").max(),
    )
    .sort("station_id")
    .collect()
)

shape: (2, 5)
┌────────────┬───────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name      ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---               ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str               ┆ u32   ┆ date       ┆ date       │
╞════════════╪═══════════════════╪═══════╪════════════╪════════════╡
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 396   ┆ 1962-09-01 ┆ 1970-12-31 │
│ 79176001   ┆ MENIGOUTE_SAPC    ┆ 31    ┆ 1964-01-01 ┆ 1964-01-31 │
└────────────┴───────────────────┴───────┴────────────┴────────────┘


#### 2.5.2 `autres-parametres`

In [ ]:
with pl.Config(tbl_rows=30):
    print(
        lf_ap_profile
        .filter(
            pl.col("all_measure_count") == 0
        )
        .select(
            "station_id",
            "station_name",
            "obs_date"
        )
        .group_by(
            "station_id",
            "station_name",
        )
        .agg(
            count=pl.len(),
            min_date=pl.col("obs_date").min(),
            max_date=pl.col("obs_date").max(),
        )
        .sort("station_id")
        .collect()
    )

shape: (28, 5)
┌────────────┬───────────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name              ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                       ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                       ┆ u32   ┆ date       ┆ date       │
╞════════════╪═══════════════════════════╪═══════╪════════════╪════════════╡
│ 79013002   ┆ SANZAY                    ┆ 1     ┆ 1984-01-20 ┆ 1984-01-20 │
│ 79043001   ┆ BOUILLE-LORETZ            ┆ 1     ┆ 2000-08-05 ┆ 2000-08-05 │
│ 79047001   ┆ BOUSSAIS ECOLE            ┆ 2     ┆ 1951-05-06 ┆ 1951-05-28 │
│ 79049001   ┆ BRESSUIRE                 ┆ 3     ┆ 1984-01-19 ┆ 2002-08-13 │
│ 79061001   ┆ CELLES-SUR-BELL           ┆ 2     ┆ 1960-01-19 ┆ 1967-11-02 │
│ 79079001   ┆ MAULEON                   ┆ 5     ┆ 1956-01-23 ┆ 1984-01-26 │
│ 79103001   ┆ COURLAY PLACE DE L'EGLISE ┆ 10    ┆ 1957-11-03 ┆ 1963-02-18 │
│ 79134001   ┆ GLENAY_SAPC               ┆ 3     ┆ 1952-11-27

### 2.6 Audit of parameters with uncertain types

In [ ]:
print(dwl.audit_numeric_types_lf(lf_ap_wide, dwl.FIELDS_TO_AUDIT))

shape: (8, 5)
┌────────────┬──────┬───────┬──────────┬──────────────────┐
│ column     ┆ min  ┆ max   ┆ n_unique ┆ fractional_count │
│ ---        ┆ ---  ┆ ---   ┆ ---      ┆ ---              │
│ str        ┆ f64  ┆ f64   ┆ i64      ┆ i64              │
╞════════════╪══════╪═══════╪══════════╪══════════════════╡
│ hneigef    ┆ 0.0  ┆ 18.0  ┆ 16       ┆ 0                │
│ neigetotx  ┆ 0.0  ┆ 18.0  ┆ 17       ┆ 0                │
│ neigetot06 ┆ 0.0  ┆ 18.0  ┆ 15       ┆ 0                │
│ uv_indicex ┆ null ┆ null  ┆ 0        ┆ 0                │
│ sigma      ┆ 0.0  ┆ 101.0 ┆ 102      ┆ 0                │
│ un         ┆ 1.0  ┆ 100.0 ┆ 100      ┆ 0                │
│ ux         ┆ 34.0 ┆ 100.0 ┆ 62       ┆ 0                │
│ um         ┆ 20.0 ┆ 100.0 ┆ 78       ┆ 0                │
└────────────┴──────┴───────┴──────────┴──────────────────┘


## 3. Wide -> long transformation

For `RR-T-Vent`, the two columns `status_fxi3s` and `status_dxi3s` are attributes of the parameters `fxi3s` and `dxi3s` (codes 0/1). Therefore, a column named `status_code` is included in the long representation of the `RR-T-Vent` data; for the relevant parameters, this column can take the values 0 or 1 (or null), and null for the other parameters.

Target grain?

### `RR-T-Vent`

In [11]:
value_columns_rtv = [
    param.lower()
    for param in dwl.RR_T_VENT_ALL_PARAMS
]

quality_columns_rtv = [
    f"q{param.lower()}"
    for param in dwl.RR_T_VENT_ALL_PARAMS
]

lf_long_rtv_all_values = (
    lf_rtv_wide_all
    .unpivot(
        on=value_columns_rtv,
        index=["station_id", "obs_date", "source_file"],
        variable_name="param_code",
        value_name="value",
    )
    .filter(pl.col("value").is_not_null())
)

lf_long_rtv_all_quality = (
    lf_rtv_wide_all
    .unpivot(
        on=quality_columns_rtv,
        index=["station_id", "obs_date", "source_file"],
        variable_name="quality_param_code",
        value_name="quality_code",
    )
    .filter(pl.col("quality_code").is_not_null())
    .with_columns(
        pl.col("quality_param_code")
        .str.slice(1)
        .alias("param_code")
    )
    .drop("quality_param_code")
)

join_keys = ["source_file", "station_id", "obs_date", "param_code"]

lf_long_rtv_all = (
    lf_long_rtv_all_values
    .join(
        lf_long_rtv_all_quality,
        on=join_keys,
        how="left",
        validate="1:1",
    )
)

lf_rtv_status = (
    lf_rtv_wide_all
    .select(
        "station_id",
        "obs_date",
        "status_fxi3s",
        "status_dxi3s",
        "source_file",
    )
)

lf_long_rtv_bench = (
    lf_long_rtv_all
    .join(
        lf_rtv_status,
        on=["source_file", "station_id", "obs_date"],
        how="left",
        validate="m:1",
    )
    .with_columns(
        pl.when(pl.col("param_code") == "fxi3s")
        .then(pl.col("status_fxi3s"))
        .when(pl.col("param_code") == "dxi3s")
        .then(pl.col("status_dxi3s"))
        .otherwise(None)
        .cast(pl.UInt8)
        .alias("status_code")
    )
    .drop(
        "status_fxi3s",
        "status_dxi3s",
        "source_file",
    )
)

# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

# print("""df_rtv_all.estimated_size("mb")""")
# print(df_rtv_all.estimated_size("mb"))
# print("""df_long_rtv_all.estimated_size("mb")""")
# print(df_long_rtv_all.estimated_size("mb"))


In [12]:
df_rtv_wide_bench = (
    lf_rtv_wide_all
    .drop(["station_name", "source_file"])
    .collect()
)
print_memory_usage()

PID: 2178955
RSS current: 479.51953125 MB
RSS peak: 523.9375 MB


In [13]:
df_rtv_long_bench = (
    lf_long_rtv_bench
    .collect()
)
print_memory_usage()

PID: 2178955
RSS current: 2427.5390625 MB
RSS peak: 4054.4140625 MB


In [14]:
print_memory_usage()

PID: 2178955
RSS current: 2427.5390625 MB
RSS peak: 4054.4140625 MB


In [15]:
df_rtv_long_bench_id = dwl.replace_param_code_with_param_id(df_rtv_long_bench)
print_memory_usage()

PID: 2178955
RSS current: 2248.56640625 MB
RSS peak: 4054.4140625 MB


In [16]:
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

# print("""df_rtv_all.estimated_size("mb")""")
# print(df_rtv_wide_all.estimated_size("mb"))
# print("""df_long_rtv_all.estimated_size("mb")""")
# print(df_long_rtv_all.estimated_size("mb"))

### `autres-parametres`

In [17]:
value_columns_ap = [
    param.lower()
    for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS
]

quality_columns_ap = [
    f"q{param.lower()}"
    for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS
]

lf_long_ap_values = (
    lf_ap_wide
    .unpivot(
        on=value_columns_ap,
        index=["station_id", "obs_date", "source_file"],
        variable_name="param_code",
        value_name="value",
    )
    .filter(pl.col("value").is_not_null())
)

lf_long_ap_quality = (
    lf_ap_wide
    .unpivot(
        on=quality_columns_ap,
        index=["station_id", "obs_date", "source_file"],
        variable_name="quality_param_code",
        value_name="quality_code",
    )
    .filter(pl.col("quality_code").is_not_null())
    .with_columns(
        pl.col("quality_param_code")
        .str.slice(1)
        .alias("param_code")
    )
    .drop("quality_param_code")
)

join_keys = ["source_file", "station_id", "obs_date", "param_code"]

lf_long_ap = (
    lf_long_ap_values
    .join(
        lf_long_ap_quality,
        on=join_keys,
        how="left",
        validate="1:1",
    )
)

lf_long_ap_bench = (
    lf_long_ap
    .with_columns(
        status_code=pl.lit(None, dtype=pl.UInt8),
    )
    .drop("source_file")
)

# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

# print("""df_rtv_all.estimated_size("mb")""")
# print(df_rtv_all.estimated_size("mb"))
# print("""df_long_rtv_all.estimated_size("mb")""")
# print(df_long_rtv_all.estimated_size("mb"))


In [18]:
df_ap_wide_bench = (
    lf_ap_wide
    .drop(["station_name", "source_file"])
    .collect()
)
print_memory_usage()

PID: 2178955
RSS current: 791.22265625 MB
RSS peak: 4054.4140625 MB


In [19]:
df_ap_long_bench = (
    lf_long_ap_bench
    .collect()
)
print_memory_usage()

PID: 2178955
RSS current: 3119.37109375 MB
RSS peak: 4054.4140625 MB


In [20]:
df_ap_long_bench_id = dwl.replace_param_code_with_param_id(df_ap_long_bench)
print_memory_usage()

PID: 2178955
RSS current: 3121.01953125 MB
RSS peak: 4054.4140625 MB


### MOVE ELSEWHERE. Series of days with Tx above 20 °C

In [15]:
df_tx_20 = (
    df_rtv_wide_bench
    .filter(station_id="79191005")
    .select("station_id", "obs_date", "tx")
    .filter(pl.col("tx").ge(20.0))
    .sort("obs_date")
    .with_row_index("index")
    .with_columns(
        epoch_d=pl.col("obs_date").dt.epoch("d"),
        group_nr=pl.col("obs_date").dt.epoch("d") - pl.col("index")
    )
)
with pl.Config(tbl_rows=300):
    print(
        df_tx_20
        .filter(
            pl.col("obs_date").dt.year().eq(1976),
            pl.col("obs_date").dt.month().is_in([5, 6, 7]),
        )
    )

with pl.Config(tbl_rows=300):
    print(
        df_tx_20
        # df_rtv_wide_bench
        # .filter(station_id="79191005")
        # .select("station_id", "obs_date", "tx")
        # .filter(pl.col("tx").ge(20.0))
        # .sort("obs_date")
        # .with_row_index("index")
        # .with_columns(
        #     group_nr=pl.col("obs_date").dt.epoch("d") - pl.col("index")
        # )
        .group_by("group_nr")
        .agg(
            min_date=pl.col("obs_date").min(),
            max_date=pl.col("obs_date").max(),
            length=pl.len(),
            txn=pl.col("tx").min(),
            txx=pl.col("tx").max(),
            txm=pl.col("tx").mean(),
        )
        .filter(pl.col("length").ge(10))
        .sort("length", descending=True)
    )

shape: (79, 6)
┌───────┬────────────┬────────────┬──────┬─────────┬──────────┐
│ index ┆ station_id ┆ obs_date   ┆ tx   ┆ epoch_d ┆ group_nr │
│ ---   ┆ ---        ┆ ---        ┆ ---  ┆ ---     ┆ ---      │
│ u32   ┆ str        ┆ date       ┆ f64  ┆ i32     ┆ i64      │
╞═══════╪════════════╪════════════╪══════╪═════════╪══════════╡
│ 2325  ┆ 79191005   ┆ 1976-05-05 ┆ 27.1 ┆ 2316    ┆ -9       │
│ 2326  ┆ 79191005   ┆ 1976-05-06 ┆ 29.0 ┆ 2317    ┆ -9       │
│ 2327  ┆ 79191005   ┆ 1976-05-07 ┆ 29.4 ┆ 2318    ┆ -9       │
│ 2328  ┆ 79191005   ┆ 1976-05-08 ┆ 27.2 ┆ 2319    ┆ -9       │
│ 2329  ┆ 79191005   ┆ 1976-05-09 ┆ 24.0 ┆ 2320    ┆ -9       │
│ 2330  ┆ 79191005   ┆ 1976-05-10 ┆ 21.2 ┆ 2321    ┆ -9       │
│ 2331  ┆ 79191005   ┆ 1976-05-14 ┆ 22.0 ┆ 2325    ┆ -6       │
│ 2332  ┆ 79191005   ┆ 1976-05-16 ┆ 22.0 ┆ 2327    ┆ -5       │
│ 2333  ┆ 79191005   ┆ 1976-05-17 ┆ 25.6 ┆ 2328    ┆ -5       │
│ 2334  ┆ 79191005   ┆ 1976-05-22 ┆ 23.1 ┆ 2333    ┆ -1       │
│ 2335  ┆ 79191005   ┆ 19

### Comparisons

In [21]:
df_daily_long = pl.concat(
    [df_rtv_long_bench, df_ap_long_bench],
    how="vertical",
)
print_memory_usage()

PID: 2178955
RSS current: 840.32421875 MB
RSS peak: 4054.4140625 MB


In [22]:
df_daily_long_id = pl.concat(
    [df_rtv_long_bench_id, df_ap_long_bench_id],
    how="vertical",
)
print_memory_usage()

PID: 2178955
RSS current: 840.32421875 MB
RSS peak: 4054.4140625 MB


In [23]:
wide_long_comparison = pl.DataFrame({
    "name": [
        "df_rtv_wide_bench",
        "df_ap_wide_bench",
        "df_rtv_wide_bench + df_ap_wide_bench",
        "df_rtv_long_bench (param_code)",
        "df_ap_long_bench (param_code)",
        "daily_long (param_code)",
        "df_rtv_long_bench_id",
        "df_ap_long_bench_id",
        "daily_long_id",
    ],
    "height": [
        df_rtv_wide_bench.height,
        df_ap_wide_bench.height,
        None,
        df_rtv_long_bench.height,
        df_ap_long_bench.height,
        df_daily_long.height,
        df_rtv_long_bench_id.height,
        df_ap_long_bench_id.height,
        df_daily_long_id.height,
    ],
    "width": [
        df_rtv_wide_bench.width,
        df_ap_wide_bench.width,
        None,
        df_rtv_long_bench.width,
        df_ap_long_bench.width,
        df_daily_long.width,
        df_rtv_long_bench_id.width,
        df_ap_long_bench_id.width,
        df_daily_long_id.width,
    ],
    "estimated_size": [
        df_rtv_wide_bench.estimated_size("mb"),
        df_ap_wide_bench.estimated_size("mb"),
        df_rtv_wide_bench.estimated_size("mb") + df_ap_wide_bench.estimated_size("mb"),
        df_rtv_long_bench.estimated_size("mb"),
        df_ap_long_bench.estimated_size("mb"),
        df_daily_long.estimated_size("mb"),
        df_rtv_long_bench_id.estimated_size("mb"),
        df_ap_long_bench_id.estimated_size("mb"),
        df_daily_long_id.estimated_size("mb"),
    ],
})

with pl.Config(fmt_str_lengths=50):
    print(wide_long_comparison)

shape: (9, 4)
┌──────────────────────────────────────┬─────────┬───────┬────────────────┐
│ name                                 ┆ height  ┆ width ┆ estimated_size │
│ ---                                  ┆ ---     ┆ ---   ┆ ---            │
│ str                                  ┆ i64     ┆ i64   ┆ f64            │
╞══════════════════════════════════════╪═════════╪═══════╪════════════════╡
│ df_rtv_wide_bench                    ┆ 1172316 ┆ 56    ┆ 203.755767     │
│ df_ap_wide_bench                     ┆ 565558  ┆ 84    ┆ 139.162449     │
│ df_rtv_wide_bench + df_ap_wide_bench ┆ null    ┆ null  ┆ 342.918216     │
│ df_rtv_long_bench (param_code)       ┆ 4242670 ┆ 6     ┆ 102.478333     │
│ df_ap_long_bench (param_code)        ┆ 2148378 ┆ 6     ┆ 55.614201      │
│ daily_long (param_code)              ┆ 6391048 ┆ 6     ┆ 158.092534     │
│ df_rtv_long_bench_id                 ┆ 4242670 ┆ 6     ┆ 98.118547      │
│ df_ap_long_bench_id                  ┆ 2148378 ┆ 6     ┆ 49.684685      

### Parquet

In [72]:
rtv_wide_parquet = DUMP_DIR / "rr_t_vent_wide.parquet"
ap_wide_parquet = DUMP_DIR / "autres_parametres_wide.parquet"
daily_long_parquet_param_code = DUMP_DIR / "daily_long_param_code.parquet"
daily_long_parquet_param_code_wout_status = DUMP_DIR / "daily_long_param_code_wout_status.parquet"
daily_long_parquet_param_id = DUMP_DIR / "daily_long_param_id.parquet"
df_rtv_wide_bench.write_parquet(rtv_wide_parquet, compression="zstd")
df_ap_wide_bench.write_parquet(ap_wide_parquet, compression="zstd")
df_daily_long.write_parquet(daily_long_parquet_param_code, compression="zstd")
df_daily_long.drop("status_code").write_parquet(daily_long_parquet_param_code_wout_status, compression="zstd")
df_daily_long_id.write_parquet(daily_long_parquet_param_id, compression="zstd")

In [73]:
with pl.Config(fmt_str_lengths=150):
    print(pl.DataFrame({
        "name": [
            "rr_t_vent_wide.parquet",
            "autres_parametres_wide.parquet",
            "sum of rr_t_vent_wide + autres_parametres_wide",
            "daily_long_param_code.parquet",
            "daily_long_param_code_wout_status.parquet",
            "daily_long_param_id.parquet",
        ],
        "parquet_size": [
            rtv_wide_parquet.stat().st_size,
            ap_wide_parquet.stat().st_size,
            rtv_wide_parquet.stat().st_size + ap_wide_parquet.stat().st_size,
            daily_long_parquet_param_code.stat().st_size,
            daily_long_parquet_param_code_wout_status.stat().st_size,
            daily_long_parquet_param_id.stat().st_size,
        ]
    }))

shape: (6, 2)
┌────────────────────────────────────────────────┬──────────────┐
│ name                                           ┆ parquet_size │
│ ---                                            ┆ ---          │
│ str                                            ┆ i64          │
╞════════════════════════════════════════════════╪══════════════╡
│ rr_t_vent_wide.parquet                         ┆ 9012547      │
│ autres_parametres_wide.parquet                 ┆ 3040501      │
│ sum of rr_t_vent_wide + autres_parametres_wide ┆ 12053048     │
│ daily_long_param_code.parquet                  ┆ 18993315     │
│ daily_long_param_code_wout_status.parquet      ┆ 18974730     │
│ daily_long_param_id.parquet                    ┆ 18992912     │
└────────────────────────────────────────────────┴──────────────┘


## 4. Generating benchmark files for PostgreSQL import

### Stations

In [39]:
rtv_stations_id_name_df = (
    lf_rtv_wide_all
    .select("station_id", "station_name")
    .unique()
    .collect()
)

ap_stations_id_name_df = (
    lf_ap_wide
    .select("station_id", "station_name")
    .unique()
    .collect()
)

stations_id_name = pl.concat([rtv_stations_id_name_df, ap_stations_id_name_df]).unique()

In [40]:
rtv_stations_id_only_df = (
    lf_rtv_wide_all
    .select("station_id")
    .unique()
    .collect()
)

ap_stations_id_only_df = (
    lf_ap_wide
    .select("station_id")
    .unique()
    .collect()
)

stations_id_only = pl.concat([rtv_stations_id_only_df, ap_stations_id_only_df]).unique()

In [41]:
stations_check_multi_name = (
    stations_id_name
    .join(
        stations_id_only,
        how="left",
        on="station_id",
        validate="1:1",
    )
)
print(stations_check_multi_name)

shape: (121, 2)
┌────────────┬──────────────────────────────┐
│ station_id ┆ station_name                 │
│ ---        ┆ ---                          │
│ str        ┆ str                          │
╞════════════╪══════════════════════════════╡
│ 79201001   ┆ PAMPROUX BOURG               │
│ 79320001   ┆ SURIN                        │
│ 79220002   ┆ PRIN-DEYRANCON_SAPC          │
│ 79101001   ┆ COULONGES-SUR-L'AUTIZE       │
│ 79191002   ┆ NIORT                        │
│ …          ┆ …                            │
│ 79140001   ┆ HANC AGGLOMERATION           │
│ 79280001   ┆ SAINT-MAURICE-LA-FOUGEREUSE  │
│ 79307001   ┆ SAUZE-VAUSSAIS AGGLOMERATION │
│ 79320002   ┆ SURIN_SAPC                   │
│ 79216001   ┆ PRAHECQ AGGLOMERATION        │
└────────────┴──────────────────────────────┘


In [ ]:
(
    stations_check_multi_name
    .sort("station_id")
    .write_csv(DUMP_DIR / "stations.csv", separator=";")
)

### Parameters

In [ ]:
(
    pl.DataFrame({
        "param_id": dwl.PARAM_IDS.values(),
        "param_code": dwl.PARAM_IDS.keys(),
    })
    .write_csv(DUMP_DIR / "parameters.csv", separator=";")
)

### `daily_rr_t_vent_wide` table

In [ ]:
(
    df_rtv_wide_bench
    .write_csv(DUMP_DIR / "daily_rr_t_vent_wide.csv", separator=";")
)

### `daily_autres_parametres_wide` table

In [ ]:
(
    df_ap_wide_bench
    .write_csv(DUMP_DIR / "daily_autres_parametres_wide.csv", separator=";")
)

### `daily_long` table

In [ ]:
(
    df_daily_long_id
    .select(
        "station_id",
        "obs_date",
        "param_id", 
        "value",
        "quality_code",
        "status_code"
    )
    .write_csv(DUMP_DIR / "daily_long.csv", separator=";")
)

## 5. PostgreSQL

In [62]:
SCHEMA = "spike"

In [63]:
schema_filepath = config.PROJECT_ROOT / "sql" / "spikes" / "daily_wide_long" / "schema.sql"
init_spike_schema_query = schema_filepath.read_text()

print(init_spike_schema_query)

DROP SCHEMA IF EXISTS spike CASCADE;
CREATE SCHEMA spike;

CREATE TABLE spike.stations (
    station_id VARCHAR(8) PRIMARY KEY,
    station_name TEXT NOT NULL
);

CREATE TABLE spike.parameters (
    param_id SMALLINT PRIMARY KEY,
    param_code TEXT NOT NULL UNIQUE
);

CREATE TABLE spike.daily_rr_t_vent_wide (
    station_id VARCHAR(8) NOT NULL,
    obs_date DATE NOT NULL,
    rr DOUBLE PRECISION,
    qrr SMALLINT,
    tn DOUBLE PRECISION,
    qtn SMALLINT,
    htn SMALLINT,
    qhtn SMALLINT,
    tx DOUBLE PRECISION,
    qtx SMALLINT,
    htx SMALLINT,
    qhtx SMALLINT,
    tm DOUBLE PRECISION,
    qtm SMALLINT,
    tntxm DOUBLE PRECISION,
    qtntxm SMALLINT,
    tampli DOUBLE PRECISION,
    qtampli SMALLINT,
    tnsol DOUBLE PRECISION,
    qtnsol SMALLINT,
    tn50 DOUBLE PRECISION,
    qtn50 SMALLINT,
    dg SMALLINT,
    qdg SMALLINT,
    ffm DOUBLE PRECISION,
    qffm SMALLINT,
    ff2m DOUBLE PRECISION,
    qff2m SMALLINT,
    fxy DOUBLE PRECISION,
    qfxy SMALLINT,
    dxy SM

In [ ]:
schema_filepath = config.PROJECT_ROOT / "sql" / "spikes" / "daily_wide_long" / "schema.sql"
init_spike_schema_query = schema_filepath.read_text()

with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    with conn.cursor() as cur:
        cur.execute(init_spike_schema_query)

    with conn.cursor() as cur:
        check_query = psycopg.sql.SQL(
            """
            SELECT
                c.relname AS table_name
            FROM pg_class AS c
            JOIN pg_namespace AS n
                ON n.oid = c.relnamespace
            WHERE n.nspname = %s
            """
        )
        # ).format(
        #     psycopg.sql.Identifier(SCHEMA)
        # )
        cur.execute(check_query, SCHEMA)
        print(cur.fetchall())

[('stations',), ('stations_pkey',), ('parameters',), ('parameters_pkey',), ('parameters_param_code_key',), ('daily_rr_t_vent_wide',), ('daily_rr_t_vent_wide_pkey',), ('daily_autres_parametres_wide',), ('daily_autres_parametres_wide_pkey',), ('daily_long',), ('daily_long_pkey',)]


In [74]:
schema_filepath = config.PROJECT_ROOT / "sql" / "spikes" / "daily_wide_long" / "schema.sql"
init_spike_schema_query = schema_filepath.read_text()

with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    with conn.cursor() as cur:
        check_query = psycopg.sql.SQL(
            """
            SELECT
                c.relname AS table_name
            FROM pg_class AS c
            JOIN pg_namespace AS n
                ON n.oid = c.relnamespace
            WHERE n.nspname = %s
            """
        )
        # ).format(
        #     psycopg.sql.Identifier(SCHEMA)
        # )
        cur.execute(check_query, (SCHEMA,))
        print(cur.fetchall())

[('stations',), ('stations_pkey',), ('parameters',), ('parameters_pkey',), ('parameters_param_code_key',), ('daily_rr_t_vent_wide',), ('daily_rr_t_vent_wide_pkey',), ('daily_autres_parametres_wide',), ('daily_autres_parametres_wide_pkey',), ('daily_long',), ('daily_long_pkey',)]


In [ ]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    check_query = """
        SELECT
            c.relname AS table_name
        FROM pg_class AS c
        JOIN pg_namespace AS n
            ON n.oid = c.relnamespace
        WHERE n.nspname = 'spike'
        """
    
    with conn.cursor() as cur:
        ret = pl.read_database(
            query=check_query,
            connection=conn,
        )

with pl.Config(tbl_rows=20, fmt_str_lengths=100):
    print(ret)

shape: (11, 1)
┌───────────────────────────────────┐
│ table_name                        │
│ ---                               │
│ str                               │
╞═══════════════════════════════════╡
│ stations                          │
│ stations_pkey                     │
│ parameters                        │
│ parameters_pkey                   │
│ parameters_param_code_key         │
│ daily_rr_t_vent_wide              │
│ daily_rr_t_vent_wide_pkey         │
│ daily_autres_parametres_wide      │
│ daily_autres_parametres_wide_pkey │
│ daily_long                        │
│ daily_long_pkey                   │
└───────────────────────────────────┘


In [72]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    check_query = psycopg.sql.SQL(
        """
        SELECT COUNT(*) AS stations FROM spike.stations;
        SELECT COUNT(*) AS parameters FROM spike.parameters;
        """
    )

    with conn.cursor() as cur:
        cur.execute(check_query)

        for _ in cur.results():
            print(cur.fetchone())

(0,)
(0,)


In [ ]:
def populate_spike_table_print(
        conn: psycopg.Connection,
        table: str,
        filename: str | None = None,
) -> None:
    if filename is None:
        filename = f"{table}.csv"
    filepath = DUMP_DIR / filename
    size_mib = filepath.stat().st_size / 1024**2
    elapsed = loader.copy_csv(conn, table, filepath)
    count = loader.get_table_count(conn, table)
    print(
        f"Loaded {SCHEMA}.{table}: "
        f"{count} rows in {elapsed:.3f} s "
        f"({size_mib / elapsed:.1f} MiB/s)"
    )


In [75]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    loader.populate_spike_table_print(conn, "stations")
    loader.populate_spike_table_print(conn, "parameters")
    loader.populate_spike_table_print(conn, "daily_rr_t_vent_wide")
    loader.populate_spike_table_print(conn, "daily_autres_parametres_wide")
    loader.populate_spike_table_print(conn, "daily_long")

#     table = "parameters"
#     elapsed = copy_csv(conn, table, DUMP_DIR / "parameters.csv")
#     count = get_table_count(conn, table)
#     print(f"Loaded spike.{table}: "
#             f"{count} rows in {elapsed:.3f} s")

#     table = "daily_rr_t_vent_wide"
#     elapsed = copy_csv(conn, table, DUMP_DIR / "daily_rr_t_vent_wide.csv")
#     count = get_table_count(conn, table)
#     print(f"Loaded spike.{table}: "
#             f"{count} rows in {elapsed:.3f} s")

#     table = "daily_autres_parametres_wide"
#     elapsed = copy_csv(conn, table, DUMP_DIR / "daily_autres_parametres_wide.csv")
#     count = get_table_count(conn, table)
#     print(f"Loaded spike.{table}: "
#             f"{count} rows in {elapsed:.3f} s")

#     table = "daily_long"
#     elapsed = copy_csv(conn, table, DUMP_DIR / "daily_long.csv")
#     count = get_table_count(conn, table)
#     print(f"Loaded spike.{table}: "
#             f"{count} rows in {elapsed:.3f} s")

Loaded spike.stations: 121 rows in 0.077 s (Input file: 0.00307464599609375 MiB -> Rate: 0.0 MiB/s)
Loaded spike.parameters: 67 rows in 0.010 s (Input file: 0.0005693435668945312 MiB -> Rate: 0.1 MiB/s)
Loaded spike.daily_rr_t_vent_wide: 1172316 rows in 14.159 s (Input file: 100.30730247497559 MiB -> Rate: 7.1 MiB/s)
Loaded spike.daily_autres_parametres_wide: 565558 rows in 5.018 s (Input file: 61.392518043518066 MiB -> Rate: 12.2 MiB/s)
Loaded spike.daily_long: 6391048 rows in 36.082 s (Input file: 183.48267269134521 MiB -> Rate: 5.1 MiB/s)


In [3]:
SIZE_QUERY = """
SELECT
    c.relname AS table_name,
    pg_relation_size(c.oid) AS heap_bytes,
    pg_indexes_size(c.oid) AS index_bytes,
    pg_total_relation_size(c.oid) AS total_bytes
FROM pg_class AS c
JOIN pg_namespace AS n
    ON n.oid = c.relnamespace
WHERE n.nspname = 'spike'
  AND c.relname IN (
      'daily_rr_t_vent_wide',
      'daily_autres_parametres_wide',
      'daily_long'
  )
ORDER BY c.relname;
"""

with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    pg_sizes = pl.read_database(
        query=SIZE_QUERY,
        connection=conn,
    )

with pl.Config(tbl_rows=20, fmt_str_lengths=150):
    print(pg_sizes)

shape: (3, 4)
┌──────────────────────────────┬────────────┬─────────────┬─────────────┐
│ table_name                   ┆ heap_bytes ┆ index_bytes ┆ total_bytes │
│ ---                          ┆ ---        ┆ ---         ┆ ---         │
│ str                          ┆ i64        ┆ i64         ┆ i64         │
╞══════════════════════════════╪════════════╪═════════════╪═════════════╡
│ daily_autres_parametres_wide ┆ 51642368   ┆ 18046976    ┆ 69730304    │
│ daily_long                   ┆ 436797440  ┆ 330145792   ┆ 767090688   │
│ daily_rr_t_vent_wide         ┆ 117964800  ┆ 37388288    ┆ 155410432   │
└──────────────────────────────┴────────────┴─────────────┴─────────────┘


## 6. Analytical requests

In [62]:
def print_dict(var_dict):
    print(json.dumps(var_dict, indent=4))

### 6.1 Série temporelle d'un paramètre pour une station

In [52]:
param_id_tx = dwl.PARAM_IDS["tx"]
print(param_id_tx)

3


#### 6.1.1 Polars (baseline in-memory)

##### 6.1.1.1 Polars wide

In [76]:
def first_query_polars_wide():
    return (
        df_rtv_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("tx").is_not_null(),
        )
        .select("station_id", "obs_date", "tx")
        .sort("obs_date")
        .collect()
    )

In [111]:
print_dict(dwl.benchmark(first_query_polars_wide))

{
    "median_s": 0.006343893997836858,
    "min_s": 0.005693519022315741,
    "max_s": 0.008508072001859546,
    "runs_s": [
        0.005693519022315741,
        0.008508072001859546,
        0.006343893997836858,
        0.005830100039020181,
        0.0075113350176252425,
        0.006600042979698628,
        0.005909001978579909
    ]
}


##### 6.1.1.2 Polars long

In [112]:
def first_query_polars_long():
    return (
        df_rtv_long_bench_id
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("param_id") == param_id_tx,
        )
        .select(
            "station_id",
            "obs_date",
            pl.col("value").alias("tx"),
        )
        .collect()
    )

In [113]:
print_dict(dwl.benchmark(first_query_polars_long))

{
    "median_s": 0.00805644498905167,
    "min_s": 0.007364261953625828,
    "max_s": 0.008778417017310858,
    "runs_s": [
        0.007364261953625828,
        0.008269544981885701,
        0.00805644498905167,
        0.008778417017310858,
        0.007919236028101295,
        0.00829951703781262,
        0.007631803979165852
    ]
}


#### 6.1.2 PostgreSQL + psycopg/Polars

##### 6.1.2.1 PostgreSQL wide

In [ ]:
FIRST_QUERY_PG_WIDE = """
    SELECT station_id, obs_date, tx
    FROM spike.daily_rr_t_vent_wide
    WHERE station_id = '79191005' AND tx IS NOT NULL
    ORDER BY obs_date
"""

In [114]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    def first_query_pg_wide():
        return pl.read_database(
            query=FIRST_QUERY_PG_WIDE,
            connection=conn,
        )
    
    first_query_wide_pg_benchmark = dwl.benchmark(first_query_pg_wide)

print_dict(first_query_wide_pg_benchmark)

{
    "median_s": 0.029222233046311885,
    "min_s": 0.027888866025023162,
    "max_s": 0.03201542003080249,
    "runs_s": [
        0.028861441009212285,
        0.029222233046311885,
        0.02876418095547706,
        0.03201542003080249,
        0.03057058999547735,
        0.02958502300316468,
        0.027888866025023162
    ]
}


In [115]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "EXPLAIN (ANALYZE, BUFFERS) " + FIRST_QUERY_PG_WIDE
        )
        print("\n".join(row[0] for row in cur.fetchall()))

Sort  (cost=17566.86..17631.73 rows=25947 width=21) (actual time=15.279..16.409 rows=24788 loops=1)
  Sort Key: obs_date
  Sort Method: quicksort  Memory: 1737kB
  Buffers: shared hit=766
  ->  Bitmap Heap Scan on daily_rr_t_vent_wide  (cost=609.52..15664.52 rows=25947 width=21) (actual time=1.224..9.122 rows=24788 loops=1)
        Recheck Cond: ((station_id)::text = '79191005'::text)
        Heap Blocks: exact=664
        Buffers: shared hit=763
        ->  Bitmap Index Scan on daily_rr_t_vent_wide_pkey  (cost=0.00..603.03 rows=25947 width=0) (actual time=1.157..1.158 rows=24788 loops=1)
              Index Cond: ((station_id)::text = '79191005'::text)
              Buffers: shared hit=99
Planning:
  Buffers: shared hit=91
Planning Time: 0.397 ms
Execution Time: 17.314 ms


##### 6.1.2.2 PostgreSQL long

In [116]:
FIRST_QUERY_PG_LONG = """
    SELECT station_id, obs_date, value AS tx
    FROM spike.daily_long
    WHERE param_id = 3 AND station_id = '79191005'
    ORDER BY obs_date
"""

In [117]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    def first_query_pg_long():
        return pl.read_database(
            query=FIRST_QUERY_PG_LONG,
            connection=conn,
        )
    
    first_query_long_pg_benchmark = dwl.benchmark(first_query_pg_long)

print_dict(first_query_long_pg_benchmark)

{
    "median_s": 0.052299617032986134,
    "min_s": 0.04867212800309062,
    "max_s": 0.05761066300328821,
    "runs_s": [
        0.052299617032986134,
        0.05104215600294992,
        0.05268220597645268,
        0.05761066300328821,
        0.054539784032385796,
        0.04867212800309062,
        0.05158952402416617
    ]
}


In [118]:
with psycopg.connect(
    host=os.getenv("POSTGRES_LOCAL_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_LOCAL_PORT", "5433"),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "EXPLAIN (ANALYZE, BUFFERS) " + FIRST_QUERY_PG_LONG
        )
        print("\n".join(row[0] for row in cur.fetchall()))

Sort  (cost=84923.10..85045.58 rows=48990 width=21) (actual time=36.959..37.978 rows=24775 loops=1)
  Sort Key: obs_date
  Sort Method: quicksort  Memory: 1736kB
  Buffers: shared hit=4855
  ->  Bitmap Heap Scan on daily_long  (cost=26002.69..81106.73 rows=48990 width=21) (actual time=30.284..32.880 rows=24775 loops=1)
        Recheck Cond: (((station_id)::text = '79191005'::text) AND (param_id = 3))
        Heap Blocks: exact=209
        Buffers: shared hit=4852
        ->  Bitmap Index Scan on daily_long_pkey  (cost=0.00..25990.44 rows=48990 width=0) (actual time=30.247..30.247 rows=24775 loops=1)
              Index Cond: (((station_id)::text = '79191005'::text) AND (param_id = 3))
              Buffers: shared hit=4643
Planning:
  Buffers: shared hit=115
Planning Time: 0.376 ms
Execution Time: 38.905 ms


#### 6.1.3. Parquet + Polars

##### 6.1.3.1. Parquet wide

In [119]:
def first_query_parquet_wide():
    return (
        pl.scan_parquet(rtv_wide_parquet)
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("tx").is_not_null(),
        )
        .select("station_id", "obs_date", "tx")
        .sort("obs_date")
        .collect()
    )

In [120]:
print_dict(dwl.benchmark(first_query_parquet_wide))

{
    "median_s": 0.007052152999676764,
    "min_s": 0.006274043989833444,
    "max_s": 0.008581664005760103,
    "runs_s": [
        0.007915547001175582,
        0.008581664005760103,
        0.006714863004162908,
        0.007052152999676764,
        0.007797632017172873,
        0.0069220139994286,
        0.006274043989833444
    ]
}


##### 6.1.3.2 Parquet long

In [121]:
def first_query_parquet_long():
    return (
        pl.scan_parquet(daily_long_parquet_param_id)
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("param_id") == param_id_tx,
        )
        .select(
            "station_id",
            "obs_date",
            pl.col("value").alias("tx"),
        )
        .sort("obs_date")
        .collect()
    )

In [122]:
print_dict(dwl.benchmark(first_query_parquet_long))

{
    "median_s": 0.006640364008489996,
    "min_s": 0.006036065984517336,
    "max_s": 0.007332895998843014,
    "runs_s": [
        0.007332895998843014,
        0.007088451005984098,
        0.006862865993753076,
        0.006036065984517336,
        0.006640364008489996,
        0.00656781397992745,
        0.0065595040214248
    ]
}


### 6.2. Série temporelle de plusieurs paramètres RTV pour une station et une période

#### 6.2.1 Polars (baseline in-memory)

##### 6.2.1.1. Polars wide

In [126]:
def second_query_polars_wide():
    return (
        df_rtv_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("obs_date").dt.year().is_between(1991, 2020),
            ~(
                (pl.col("rr").is_null()) &
                (pl.col("tn").is_null()) &
                (pl.col("tx").is_null()) &
                (pl.col("ffm").is_null()) &
                (pl.col("fxy").is_null()) &
                (pl.col("dxy").is_null())
            )
        )
        .select(
            "station_id",
            "obs_date",
            "rr",
            "tn",
            "tx",
            "ffm",
            "fxy",
            "dxy",
        )
        .sort("obs_date")
        .collect()
    )

In [127]:
with pl.Config():
    print(second_query_polars_wide())

shape: (10_958, 8)
┌────────────┬────────────┬──────┬──────┬──────┬─────┬──────┬─────┐
│ station_id ┆ obs_date   ┆ rr   ┆ tn   ┆ tx   ┆ ffm ┆ fxy  ┆ dxy │
│ ---        ┆ ---        ┆ ---  ┆ ---  ┆ ---  ┆ --- ┆ ---  ┆ --- │
│ str        ┆ date       ┆ f64  ┆ f64  ┆ f64  ┆ f64 ┆ f64  ┆ u16 │
╞════════════╪════════════╪══════╪══════╪══════╪═════╪══════╪═════╡
│ 79191005   ┆ 1991-01-01 ┆ 13.6 ┆ 1.8  ┆ 10.9 ┆ 4.3 ┆ 8.0  ┆ 220 │
│ 79191005   ┆ 1991-01-02 ┆ 8.4  ┆ 7.8  ┆ 11.8 ┆ 8.4 ┆ 11.0 ┆ 220 │
│ 79191005   ┆ 1991-01-03 ┆ 11.8 ┆ 9.6  ┆ 11.7 ┆ 7.3 ┆ 10.0 ┆ 220 │
│ 79191005   ┆ 1991-01-04 ┆ 4.8  ┆ 4.8  ┆ 9.8  ┆ 4.4 ┆ 10.0 ┆ 260 │
│ 79191005   ┆ 1991-01-05 ┆ 17.8 ┆ 1.4  ┆ 10.0 ┆ 6.6 ┆ 14.0 ┆ 230 │
│ …          ┆ …          ┆ …    ┆ …    ┆ …    ┆ …   ┆ …    ┆ …   │
│ 79191005   ┆ 2020-12-27 ┆ 21.4 ┆ 3.6  ┆ 10.7 ┆ 7.5 ┆ 15.5 ┆ 220 │
│ 79191005   ┆ 2020-12-28 ┆ 3.7  ┆ 1.4  ┆ 9.9  ┆ 5.9 ┆ 12.9 ┆ 250 │
│ 79191005   ┆ 2020-12-29 ┆ 0.6  ┆ 2.9  ┆ 8.2  ┆ 3.6 ┆ 7.1  ┆ 260 │
│ 79191005   ┆ 2020-12-30 ┆ 3

In [128]:
print_dict(dwl.benchmark(second_query_polars_wide))

{
    "median_s": 0.023758513038046658,
    "min_s": 0.010882824019063264,
    "max_s": 0.02729664504295215,
    "runs_s": [
        0.023127678025048226,
        0.010882824019063264,
        0.011821121035609394,
        0.026427905017044395,
        0.023758513038046658,
        0.025329957017675042,
        0.02729664504295215
    ]
}


##### 6.2.1.2. Polars long

#### 6.2.2. PostgreSQL + psycopg/Polars

##### 6.2.2.1. PostgreSQL wide

##### 6.2.2.2. PostgreSQL long

#### 6.2.3. Parquet + Polars

##### 6.2.3.1. Parquet wide

##### 6.2.3.2. Parquet long

### 6.3. Une requête impliquant un paramètre RTV et un AP

#### 6.3.1 Polars (baseline in-memory)

##### 6.3.1.1. Polars wide

In [139]:
def third_query_polars_wide():
    lf_rtv_third_query = (
        df_rtv_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("tx").is_not_null(),
            pl.col("tx") >= 30,
        )
        .select(
            "station_id",
            "obs_date",
            "tx",
        )
    )

    lf_ap_third_query = (
        df_ap_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("inst").is_not_null(),
            pl.col("inst") <= 180,
        )
        .select(
            "station_id",
            "obs_date",
            "inst",
        )
    )

    return (
        lf_rtv_third_query
        .join(
            lf_ap_third_query,
            how="inner",
            on=["station_id", "obs_date"],
        )
        .sort("obs_date")
        .collect()
    )

In [140]:
with pl.Config():
    print(third_query_polars_wide())

shape: (9, 4)
┌────────────┬────────────┬──────┬──────┐
│ station_id ┆ obs_date   ┆ tx   ┆ inst │
│ ---        ┆ ---        ┆ ---  ┆ ---  │
│ str        ┆ date       ┆ f64  ┆ u16  │
╞════════════╪════════════╪══════╪══════╡
│ 79191005   ┆ 1969-08-09 ┆ 32.2 ┆ 180  │
│ 79191005   ┆ 1977-07-03 ┆ 30.0 ┆ 156  │
│ 79191005   ┆ 1992-08-08 ┆ 33.3 ┆ 170  │
│ 79191005   ┆ 1994-07-23 ┆ 31.5 ┆ 169  │
│ 79191005   ┆ 2012-08-14 ┆ 30.2 ┆ 169  │
│ 79191005   ┆ 2012-09-23 ┆ 30.1 ┆ 178  │
│ 79191005   ┆ 2023-08-18 ┆ 33.1 ┆ 176  │
│ 79191005   ┆ 2025-07-18 ┆ 34.5 ┆ 178  │
│ 79191005   ┆ 2025-08-13 ┆ 32.4 ┆ 95   │
└────────────┴────────────┴──────┴──────┘


In [154]:
def third_query_polars_wide_full():
    lf_rtv_third_query = (
        df_rtv_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("tx").is_not_null(),
            pl.col("tx") >= 30,
        )
        .select(
            "station_id",
            "obs_date",
            "tx",
        )
    )

    lf_ap_third_query = (
        df_ap_wide_bench
        .lazy()
        .filter(
            pl.col("station_id") == "79191005",
            pl.col("inst").is_not_null(),
            pl.col("inst") <= 180,
        )
        .select(
            "station_id",
            "obs_date",
            "inst",
        )
    )

    return (
        lf_rtv_third_query
        .join(
            lf_ap_third_query,
            how="full",
            on=["station_id", "obs_date"],
        )
        .sort("obs_date")
        .collect()
    )

with pl.Config():
    print(third_query_polars_wide_full())

shape: (9_396, 6)
┌────────────┬────────────┬──────┬──────────────────┬────────────────┬──────┐
│ station_id ┆ obs_date   ┆ tx   ┆ station_id_right ┆ obs_date_right ┆ inst │
│ ---        ┆ ---        ┆ ---  ┆ ---              ┆ ---            ┆ ---  │
│ str        ┆ date       ┆ f64  ┆ str              ┆ date           ┆ u16  │
╞════════════╪════════════╪══════╪══════════════════╪════════════════╪══════╡
│ null       ┆ null       ┆ null ┆ 79191005         ┆ 2025-01-01     ┆ 0    │
│ null       ┆ null       ┆ null ┆ 79191005         ┆ 2025-01-02     ┆ 0    │
│ null       ┆ null       ┆ null ┆ 79191005         ┆ 2025-01-03     ┆ 0    │
│ null       ┆ null       ┆ null ┆ 79191005         ┆ 2025-01-04     ┆ 0    │
│ null       ┆ null       ┆ null ┆ 79191005         ┆ 2025-01-05     ┆ 1    │
│ …          ┆ …          ┆ …    ┆ …                ┆ …              ┆ …    │
│ 79191005   ┆ 2026-08-08 ┆ 35.5 ┆ null             ┆ null           ┆ null │
│ 79191005   ┆ 2026-08-09 ┆ 33.8 ┆ null       

In [141]:
print_dict(dwl.benchmark(third_query_polars_wide))

{
    "median_s": 0.015552730008494109,
    "min_s": 0.013142891984898597,
    "max_s": 0.01786034501856193,
    "runs_s": [
        0.015552730008494109,
        0.015620846999809146,
        0.016268197970930487,
        0.013936356001067907,
        0.01437208498828113,
        0.01786034501856193,
        0.013142891984898597
    ]
}


##### 6.3.1.2. Polars long

#### 6.3.2. PostgreSQL + psycopg/Polars

##### 6.3.2.1. PostgreSQL wide

##### 6.3.2.2. PostgreSQL long

#### 6.3.3. Parquet + Polars

##### 6.3.3.1. Parquet wide

##### 6.3.3.2. Parquet long

### 6.4. Agrégation d'un paramètre sur beaucoup de stations/dates

#### 6.4.1 Polars (baseline in-memory)

##### 6.4.1.1. Polars wide

In [ ]:
def fourth_query_polars_wide():
    return (
        df_rtv_wide_bench
        .lazy()
        .filter(
            pl.col("tx").is_not_null(),
        )
        .select(
            "station_id",
            "obs_date",
            "tx",
        )
        .group_by(
            "station_id",
            pl.col("obs_date").dt.year().alias("year"),
        )
        .agg(
            n_tx=pl.len(),
            n_tx_30=(pl.col("tx") >= 30).sum(),
            txx=pl.col("tx").max(),
        )
        .sort(
            # ["n_tx_30", "year"],
            # descending=[True, False],            
            ["station_id", "year"],
            descending=[False, False],
        )
        .collect()
    )

In [152]:
with pl.Config(tbl_rows=100):
    print(fourth_query_polars_wide())

shape: (1_231, 4)
┌────────────┬──────┬─────────┬──────┐
│ station_id ┆ year ┆ n_tx_30 ┆ txx  │
│ ---        ┆ ---  ┆ ---     ┆ ---  │
│ str        ┆ i32  ┆ u32     ┆ f64  │
╞════════════╪══════╪═════════╪══════╡
│ 79017001   ┆ 1974 ┆ 7       ┆ 35.0 │
│ 79017001   ┆ 1975 ┆ 10      ┆ 37.8 │
│ 79017001   ┆ 1976 ┆ 23      ┆ 35.1 │
│ 79017001   ┆ 1977 ┆ 0       ┆ 29.6 │
│ 79017001   ┆ 1978 ┆ 0       ┆ 29.8 │
│ 79017001   ┆ 1979 ┆ 1       ┆ 31.0 │
│ 79017001   ┆ 1980 ┆ 4       ┆ 32.0 │
│ 79017001   ┆ 1981 ┆ 6       ┆ 33.0 │
│ 79017001   ┆ 1982 ┆ 10      ┆ 36.0 │
│ 79017001   ┆ 1983 ┆ 11      ┆ 34.0 │
│ 79017001   ┆ 1984 ┆ 8       ┆ 34.1 │
│ 79017001   ┆ 1985 ┆ 6       ┆ 33.3 │
│ 79017001   ┆ 1986 ┆ 7       ┆ 33.8 │
│ 79017001   ┆ 1987 ┆ 14      ┆ 34.7 │
│ 79017001   ┆ 1988 ┆ 3       ┆ 31.9 │
│ 79017001   ┆ 1989 ┆ 20      ┆ 38.0 │
│ 79017001   ┆ 1990 ┆ 30      ┆ 38.9 │
│ 79017001   ┆ 1991 ┆ 23      ┆ 34.8 │
│ 79017001   ┆ 1992 ┆ 11      ┆ 33.7 │
│ 79017001   ┆ 1993 ┆ 3       ┆ 32.6 │
│ 79017

In [153]:
print_dict(dwl.benchmark(fourth_query_polars_wide))

{
    "median_s": 0.023441066034138203,
    "min_s": 0.020630737009923905,
    "max_s": 0.027322484995238483,
    "runs_s": [
        0.027322484995238483,
        0.02568775595864281,
        0.025191745022311807,
        0.022439345018938184,
        0.021573706006165594,
        0.023441066034138203,
        0.020630737009923905
    ]
}


##### 6.4.1.2. Polars long

#### 6.4.2. PostgreSQL + psycopg/Polars

##### 6.4.2.1. PostgreSQL wide

##### 6.4.2.2. PostgreSQL long

#### 6.4.3. Parquet + Polars

##### 6.4.3.1. Parquet wide

##### 6.4.3.2. Parquet long

## REARRANGE

With the scope limited to the department 79, the resulting DataFrame contains 1,172,316 lines.

### Core RR-T-Vent parameters (5 parameters)

In [ ]:
lf_rtv_wide_core = dwl.add_measure_count_lf(lf_rtv_wide_core, dwl.RR_T_VENT_CORE_PARAMS, "core_measure_count")
expected_long_rows_rtv_core = (
    lf_rtv_wide_core
    .select(pl.col("core_measure_count").sum())
    .collect()
    .item()
)
mean_measures_per_wide_row_rtv_core = (
    expected_long_rows_rtv_core / df_rtv_wide_core.height
)
measure_density_rtv_core = (
    expected_long_rows_rtv_core
    / (df_rtv_wide_core.height * len(dwl.RR_T_VENT_CORE_PARAMS))
)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print("RR-T-Vent core parameters:")
    print(f"height: {df_rtv_wide_core.height}")
    print(f"expected_long_core_rows: {expected_long_rows_rtv_core}")
    print(f"mean_measures_per_wide_row: {mean_measures_per_wide_row_rtv_core}")
    print(f"measure_density: {measure_density_rtv_core}")

    print(
        lf_rtv_wide_core
        .group_by("core_measure_count")
        .agg(
            count=pl.len(),
            pct=pl.len() / df_rtv_wide_core.height * 100
        )
        .sort("core_measure_count")
        .collect()
    )

height: 1172316
expected_long_core_rows: 2637628
mean_measures_per_wide_row: 2.249929199976798
measure_density: 0.4499858399953596
shape: (6, 3)
┌────────────────────┬────────┬───────────┐
│ core_measure_count ┆ count  ┆ pct       │
│ ---                ┆ ---    ┆ ---       │
│ u32                ┆ u32    ┆ f64       │
╞════════════════════╪════════╪═══════════╡
│ 0                  ┆ 442    ┆ 0.037703  │
│ 1                  ┆ 733868 ┆ 62.599845 │
│ 2                  ┆ 28     ┆ 0.002388  │
│ 3                  ┆ 941    ┆ 0.080268  │
│ 4                  ┆ 284304 ┆ 24.251482 │
│ 5                  ┆ 152733 ┆ 13.028313 │
└────────────────────┴────────┴───────────┘


In [ ]:
lf_rtv_wide_core = lf_rtv_wide_core.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_CORE_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_core = (
    lf_rtv_wide_core
    .group_by(
        "core_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_rtv_wide_core.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_core.head(30))

shape: (14, 4)
┌────────────────────┬───────────────────┬────────┬───────────┐
│ core_measure_count ┆ measure_signature ┆ count  ┆ pct       │
│ ---                ┆ ---               ┆ ---    ┆ ---       │
│ u32                ┆ str               ┆ u32    ┆ f64       │
╞════════════════════╪═══════════════════╪════════╪═══════════╡
│ 1                  ┆ RR                ┆ 733846 ┆ 62.597968 │
│ 4                  ┆ RR,TN,TX,TNTXM    ┆ 282433 ┆ 24.091883 │
│ 5                  ┆ RR,TN,TX,TM,TNTXM ┆ 152733 ┆ 13.028313 │
│ 4                  ┆ TN,TX,TM,TNTXM    ┆ 1871   ┆ 0.159599  │
│ 3                  ┆ TN,TX,TNTXM       ┆ 934    ┆ 0.079671  │
│ 0                  ┆                   ┆ 442    ┆ 0.037703  │
│ 2                  ┆ RR,TN             ┆ 16     ┆ 0.001365  │
│ 1                  ┆ TX                ┆ 14     ┆ 0.001194  │
│ 2                  ┆ RR,TX             ┆ 9      ┆ 0.000768  │
│ 3                  ┆ RR,TN,TM          ┆ 7      ┆ 0.000597  │
│ 1                  ┆ TM

#### Value / quality consistency

The CSV source data files contain daily observations of meteorological parameters. Each parameter corresponds to a column in the files.
Each parameter value has a corresponding quality code. In the data files, quality codes are listed in a separate column. For a given parameter, the quality code is in the column immediately to the right of the column containing the parameter itself.

In [ ]:
rtv_core_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_core, dwl.RR_T_VENT_CORE_PARAMS)

print(rtv_core_quality_audit)

shape: (10, 2)
┌─────────────────────────────┬───────┐
│ line_type                   ┆ count │
│ ---                         ┆ ---   │
│ str                         ┆ u32   │
╞═════════════════════════════╪═══════╡
│ RR_value_without_quality    ┆ 0     │
│ RR_quality_without_value    ┆ 69    │
│ TN_value_without_quality    ┆ 0     │
│ TN_quality_without_value    ┆ 1711  │
│ TX_value_without_quality    ┆ 0     │
│ TX_quality_without_value    ┆ 1705  │
│ TM_value_without_quality    ┆ 0     │
│ TM_quality_without_value    ┆ 1714  │
│ TNTXM_value_without_quality ┆ 0     │
│ TNTXM_quality_without_value ┆ 1     │
└─────────────────────────────┴───────┘


In [ ]:
orphan_quality_audit_rtv_core = pl.DataFrame()
for param in dwl.RR_T_VENT_CORE_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_core = pl.concat([
        orphan_quality_audit_rtv_core,
        lf_rtv_wide_core
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_core)

shape: (8, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
└────────────────┴────────────────────┴──────┘


### All RR-T-Vent parameters (26 parameters)

#### Measurement density

In [ ]:
lf_rtv_wide_all = dwl.add_measure_count_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS, "all_measure_count")

summary = (
    lf_rtv_wide_all
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.RR_T_VENT_ALL_PARAMS)),
    )
    .collect()
)

print("RR-T-Vent all parameters:")
print(summary)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_rtv_wide_all
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (1, 4)
┌─────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows  ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---     ┆ ---        ┆ ---                        ┆ ---             │
│ u32     ┆ u32        ┆ f64                        ┆ f64             │
╞═════════╪════════════╪════════════════════════════╪═════════════════╡
│ 1172316 ┆ 4242670    ┆ 3.61905                    ┆ 0.139194        │
└─────────┴────────────┴────────────────────────────┴─────────────────┘
RR-T-Vent all parameters:
shape: (23, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 427    ┆ 0.036424  │
│ 1                 ┆ 733810 ┆ 62.594898 │
│ 2                 ┆ 69     ┆ 0.005886  │
│ 3                 ┆ 4      ┆ 0.000341  │
│ 4                 ┆ 861    ┆ 0.073444  │
│ 5        

#### Recurring parameter signatures

In [ ]:
lf_rtv_wide_all = lf_rtv_wide_all.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_all = (
    lf_rtv_wide_all
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_rtv_wide_all.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_rtv_all_head = patterns_rtv_all.head(30)
print(f"patterns_rtv_all.shape: {patterns_rtv_all.shape}")
print(f"patterns_rtv_all sum pct: {patterns_rtv_all.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_all_head)

patterns_rtv_all.shape: (344, 4)
patterns_rtv_all sum pct: 100.00000000000006
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 1                 ┆ RR                                                      ┆ 733800 ┆ 62.594045 │
│ 5                 ┆ RR,TN,TX,TNTXM,TAMPLI                                   ┆ 280908 ┆ 23.961799 │
│ 9                 ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG                     ┆ 81036  ┆ 6.912471  │
│ 19                ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG,FFM,FXY,DXY,HXY,FXI ┆ 23138  ┆ 1.9737    

#### Value / quality consistency

In [ ]:
rtv_all_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(rtv_all_quality_audit)

shape: (52, 2)
┌──────────────────────────────┬───────┐
│ line_type                    ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ RR_value_without_quality     ┆ 0     │
│ RR_quality_without_value     ┆ 69    │
│ TN_value_without_quality     ┆ 0     │
│ TN_quality_without_value     ┆ 1711  │
│ HTN_value_without_quality    ┆ 0     │
│ HTN_quality_without_value    ┆ 2066  │
│ TX_value_without_quality     ┆ 0     │
│ TX_quality_without_value     ┆ 1705  │
│ HTX_value_without_quality    ┆ 0     │
│ HTX_quality_without_value    ┆ 2082  │
│ TM_value_without_quality     ┆ 0     │
│ TM_quality_without_value     ┆ 1714  │
│ TNTXM_value_without_quality  ┆ 0     │
│ TNTXM_quality_without_value  ┆ 1     │
│ TAMPLI_value_without_quality ┆ 0     │
│ TAMPLI_quality_without_value ┆ 1     │
│ TNSOL_value_without_quality  ┆ 0     │
│ TNSOL_quality_without_value  ┆ 24    │
│ TN50_value_without_quality   ┆ 0     │
│

In [ ]:
orphan_quality_audit_rtv_all = pl.DataFrame()
for param in dwl.RR_T_VENT_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_all = pl.concat([
        orphan_quality_audit_rtv_all,
        lf_rtv_wide_all
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_all)

shape: (39, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qhtn           ┆ 1                  ┆ 2032 │
│ qhtn           ┆ 9                  ┆ 34   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qhtx           ┆ 1                  ┆ 2051 │
│ qhtx           ┆ 9                  ┆ 31   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
│ qtampli        ┆ 1                  ┆ 1    │
│ qtnsol         ┆ 1                  ┆ 15   │
│ qtnsol         ┆ 9                  ┆ 9    │
│ qdg            ┆ 1                  ┆ 1751 

### All autres-parametres parameters (41 parameters)

#### Measurement density

In [ ]:
lf_ap_wide = dwl.add_measure_count_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS, "all_measure_count")
expected_long_core_rows_ap = (
    lf_ap_wide
    .select(pl.col("all_measure_count").sum())
    .collect()
    .item()
)
mean_measures_per_wide_row_ap = (
    expected_long_core_rows_ap / df_ap_wide.height
)
measure_density_ap = (
    expected_long_core_rows_ap
    / (df_ap_wide.height * len(dwl.AUTRES_PARAMETRES_ALL_PARAMS))
)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print("autres-parametres all parameters:")
    print(f"height: {df_ap_wide.height}")
    print(f"expected_long_core_rows: {expected_long_core_rows_ap}")
    print(f"mean_measures_per_wide_row: {mean_measures_per_wide_row_ap}")
    print(f"measure_density: {measure_density_ap}")

    print(
        lf_ap_wide
        .group_by("all_measure_count")
        .agg(
            count=pl.len(),
            pct=pl.len() / df_ap_wide.height * 100
        )
        .sort("all_measure_count")
        .collect()
    )

autres-parametres all parameters:
height: 565558
expected_long_core_rows: 2148378
mean_measures_per_wide_row: 3.7986873141216284
measure_density: 0.09265091010052752
shape: (30, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 370    ┆ 0.065422  │
│ 1                 ┆ 122836 ┆ 21.719435 │
│ 2                 ┆ 21626  ┆ 3.823834  │
│ 3                 ┆ 278177 ┆ 49.18629  │
│ 4                 ┆ 71335  ┆ 12.613207 │
│ 5                 ┆ 1198   ┆ 0.211826  │
│ 6                 ┆ 1330   ┆ 0.235166  │
│ 7                 ┆ 1741   ┆ 0.307838  │
│ 8                 ┆ 794    ┆ 0.140392  │
│ 9                 ┆ 39317  ┆ 6.951895  │
│ 10                ┆ 306    ┆ 0.054106  │
│ 11                ┆ 7693   ┆ 1.36025   │
│ 12                ┆ 1256   ┆ 0.222082  │
│ 13                ┆ 2783   ┆ 0.49208   │
│ 

#### Recurring parameter signatures

In [ ]:
lf_ap_wide = lf_ap_wide.with_columns(
    dwl.measure_signature_expr(dwl.AUTRES_PARAMETRES_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_ap = (
    lf_ap_wide
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_ap_wide.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_ap_head = patterns_ap.head(30)
print(f"patterns_ap.shape: {patterns_ap.shape}")
print(f"patterns_ap, sum pct: {patterns_ap.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_ap_head)


patterns_ap.shape: (635, 4)
patterns_ap, sum pct: 99.99999999999996
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 3                 ┆ BROU,ORAG,GRELE                                         ┆ 270391 ┆ 47.809597 │
│ 1                 ┆ ETPGRILLE                                               ┆ 114089 ┆ 20.17282  │
│ 4                 ┆ ETPGRILLE,BROU,ORAG,GRELE                               ┆ 67175  ┆ 11.87765  │
│ 9                 ┆ UN,HUN,UX,HUX,UM,DHUMI40,DHUMI80,TSVM,ETPGRILLE         ┆ 34402  ┆ 6.082842  │
│ 2     

#### Value / quality consistency

In [ ]:
ap_quality_audit = dwl.quality_pair_audit_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(ap_quality_audit)

shape: (82, 2)
┌─────────────────────────────────┬───────┐
│ line_type                       ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ DHUMEC_value_without_quality    ┆ 0     │
│ DHUMEC_quality_without_value    ┆ 151   │
│ PMERM_value_without_quality     ┆ 0     │
│ PMERM_quality_without_value     ┆ 0     │
│ PMERMIN_value_without_quality   ┆ 0     │
│ PMERMIN_quality_without_value   ┆ 2     │
│ INST_value_without_quality      ┆ 0     │
│ INST_quality_without_value      ┆ 0     │
│ GLOT_value_without_quality      ┆ 0     │
│ GLOT_quality_without_value      ┆ 5     │
│ DIFT_value_without_quality      ┆ 0     │
│ DIFT_quality_without_value      ┆ 0     │
│ DIRT_value_without_quality      ┆ 0     │
│ DIRT_quality_without_value      ┆ 0     │
│ INFRART_value_without_quality   ┆ 0     │
│ INFRART_quality_without_value   ┆ 0     │
│ UV_value_without_quality        ┆ 0     │
│ UV_quality_with

In [ ]:
orphan_quality_audit_ap = pl.DataFrame()
for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_ap = pl.concat([
        orphan_quality_audit_ap,
        lf_ap_wide
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_ap)

shape: (48, 3)
┌────────────────┬────────────────────┬───────┐
│ quality_column ┆ quality_code_value ┆ len   │
│ ---            ┆ ---                ┆ ---   │
│ str            ┆ u8                 ┆ u32   │
╞════════════════╪════════════════════╪═══════╡
│ qdhumec        ┆ 1                  ┆ 151   │
│ qpmermin       ┆ 1                  ┆ 1     │
│ qpmermin       ┆ 9                  ┆ 1     │
│ qglot          ┆ 1                  ┆ 5     │
│ qun            ┆ 1                  ┆ 44    │
│ qun            ┆ 9                  ┆ 1     │
│ qhun           ┆ 1                  ┆ 82    │
│ qhun           ┆ 9                  ┆ 3     │
│ qux            ┆ 1                  ┆ 27    │
│ qux            ┆ 9                  ┆ 1     │
│ qhux           ┆ 1                  ┆ 45    │
│ qhux           ┆ 9                  ┆ 1     │
│ qum            ┆ 1                  ┆ 28    │
│ qum            ┆ 9                  ┆ 2     │
│ qdhumi40       ┆ 1                  ┆ 54    │
│ qdhumi40       ┆ 9     